# KYC OCR Pipeline — Stage 1: Raw OCR Diagnostic Baseline

**Model:** `Qwen3.6-27B-FP8` (local Domino ModelHub)
**Hardware target:** NVIDIA H100
**Objective:** establish a *trustworthy raw OCR baseline* over the KYC document set.

This notebook deliberately does **NOT** perform structured extraction, field parsing, MRZ parsing,
normalization, validation, correction or any semantic interpretation. Its single question is:

> *What text and visible information can Qwen actually read from each scanned page?*

Pipeline: `kyc_documents.zip` -> extract -> inventory -> existence report -> readability check ->
`PDF page -> image -> Qwen -> raw text`, with per-stage timing instrumentation.

**Sections**

1. Environment and dependency inspection
2. Configuration
3. Dataset discovery
4. ZIP extraction
5. Customer/document inventory
6. Document existence report
7. PDF readability validation
8. GPU diagnostics
9. Qwen3.6 model inspection
10. Qwen3.6 model loading
11. Processor initialization
12. Warmup
13. PDF page rendering
14. Raw OCR inference (prompt + single-page call)
15. Performance instrumentation
16. Error handling / resilient drivers
17. Output generation (pipeline execution)
18. Performance summary
19. Diagnostic visualizations
20. Final run summary

## 1. Environment and dependency inspection

Nothing is installed blindly. This section only **inspects** the runtime. If something required is
missing, the notebook prints an explicit, minimal `pip install` command and stops at the dependency
gate (set `AUTO_INSTALL_MISSING = True` in the next cell only if you accept installing on Domino).

`torch` / `torchvision` / `transformers` are **never** upgraded automatically: upgrading them on a
Domino H100 image is the fastest way to break the CUDA stack.

In [ ]:
import importlib
import importlib.util
import json
import os
import platform
import sys
from datetime import datetime, timezone


def _module_version(module_name: str):
    """Import a module defensively and return its version string (or a NOT INSTALLED marker)."""
    try:
        mod = importlib.import_module(module_name)
    except Exception as exc:  # ImportError, but also broken native extensions
        return None, f"NOT INSTALLED / NOT IMPORTABLE ({type(exc).__name__}: {exc})"
    for attr in ("__version__", "VERSION", "version"):
        val = getattr(mod, attr, None)
        if isinstance(val, str):
            return mod, val
        if val is not None and not callable(val):
            return mod, str(val)
    return mod, "unknown"


ENV_REPORT = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": sys.version.replace("\n", " "),
    "python_executable": sys.executable,
    "platform": platform.platform(),
    "processor": platform.processor(),
    "cpu_count": os.cpu_count(),
    "hostname": platform.node(),
}

# ---- Core libraries -------------------------------------------------------
_torch, ENV_REPORT["torch_version"] = _module_version("torch")
_tv, ENV_REPORT["torchvision_version"] = _module_version("torchvision")
_tf, ENV_REPORT["transformers_version"] = _module_version("transformers")
_acc, ENV_REPORT["accelerate_version"] = _module_version("accelerate")
_pil, ENV_REPORT["pillow_version"] = _module_version("PIL")
_np, ENV_REPORT["numpy_version"] = _module_version("numpy")
_pd, ENV_REPORT["pandas_version"] = _module_version("pandas")
_mpl, ENV_REPORT["matplotlib_version"] = _module_version("matplotlib")

# ---- Optional / situational ----------------------------------------------
_qvu, ENV_REPORT["qwen_vl_utils_version"] = _module_version("qwen_vl_utils")
_fa, ENV_REPORT["flash_attn_version"] = _module_version("flash_attn")
_acs, ENV_REPORT["accelerate_available"] = (_acc, _acc is not None)

# ---- PyMuPDF: the import name differs between installs --------------------
# Modern PyMuPDF exposes BOTH `pymupdf` and the legacy `fitz` alias. A *different*
# PyPI package is also called `fitz`, so we validate what we imported instead of
# assuming `import fitz` is PyMuPDF.
PYMUPDF = None
PYMUPDF_IMPORT_NAME = None
ENV_REPORT["pymupdf_version"] = "NOT INSTALLED"
for _candidate in ("pymupdf", "fitz"):
    _mod, _ver = _module_version(_candidate)
    if _mod is None:
        continue
    _looks_like_pymupdf = hasattr(_mod, "open") and hasattr(_mod, "Matrix") and hasattr(_mod, "Document")
    if _looks_like_pymupdf:
        PYMUPDF = _mod
        PYMUPDF_IMPORT_NAME = _candidate
        ENV_REPORT["pymupdf_version"] = getattr(_mod, "__version__", None) or str(getattr(_mod, "version", _ver))
        break
ENV_REPORT["pymupdf_import_name"] = PYMUPDF_IMPORT_NAME

print("=" * 78)
print("ENVIRONMENT REPORT")
print("=" * 78)
for k, v in ENV_REPORT.items():
    print(f"{k:<32}: {v}")

In [ ]:
# ---- CUDA / GPU inspection (torch is the source of truth, not nvidia-smi) ----
GPU_REPORT = {"cuda_available": False, "gpu_count": 0, "devices": []}

if _torch is not None:
    torch = _torch
    GPU_REPORT["torch_cuda_build_version"] = torch.version.cuda
    GPU_REPORT["torch_cudnn_version"] = torch.backends.cudnn.version() if torch.backends.cudnn.is_available() else None
    GPU_REPORT["cuda_available"] = bool(torch.cuda.is_available())
    GPU_REPORT["bf16_supported"] = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
    if GPU_REPORT["cuda_available"]:
        GPU_REPORT["gpu_count"] = torch.cuda.device_count()
        for idx in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(idx)
            GPU_REPORT["devices"].append({
                "index": idx,
                "name": props.name,
                "total_vram_bytes": props.total_memory,
                "total_vram_gib": round(props.total_memory / (1024 ** 3), 2),
                "compute_capability": f"{props.major}.{props.minor}",
                "multi_processor_count": props.multi_processor_count,
            })
        # FP8 tensor cores require compute capability >= 8.9 (Ada) / 9.0 (Hopper H100)
        _cc = GPU_REPORT["devices"][0]["compute_capability"]
        GPU_REPORT["fp8_capable_hardware"] = float(_cc) >= 8.9
else:
    torch = None

print("=" * 78)
print("GPU / CUDA REPORT")
print("=" * 78)
print(json.dumps(GPU_REPORT, indent=2))

if not GPU_REPORT.get("cuda_available"):
    print("\n[WARNING] CUDA is NOT available. Inventory/validation sections will still run,")
    print("          but the OCR sections require a GPU (H100) on Domino.")

In [ ]:
# ---- Transformers capability probing --------------------------------------
# We do not assume which auto-classes / quantization configs exist: we probe the
# installed transformers package. Qwen3.6 may map to a different auto-class than
# older Qwen2-VL checkpoints, so the model-loading section decides *at runtime*.
TRANSFORMERS_CAPS = {}
if _tf is not None:
    import transformers

    for _cls in [
        "AutoModelForImageTextToText",
        "AutoModelForVision2Seq",
        "AutoModelForCausalLM",
        "AutoModel",
        "AutoProcessor",
        "AutoTokenizer",
        "AutoConfig",
        "BitsAndBytesConfig",
        "FineGrainedFP8Config",
        "Qwen2VLForConditionalGeneration",
        "Qwen2_5_VLForConditionalGeneration",
    ]:
        TRANSFORMERS_CAPS[_cls] = hasattr(transformers, _cls)

    # FP8Linear lives in an integrations submodule in recent releases.
    try:
        from transformers.integrations import finegrained_fp8 as _fp8mod  # noqa: F401
        TRANSFORMERS_CAPS["integrations.finegrained_fp8"] = True
        TRANSFORMERS_CAPS["FP8Linear"] = hasattr(_fp8mod, "FP8Linear")
    except Exception:
        TRANSFORMERS_CAPS["integrations.finegrained_fp8"] = False
        TRANSFORMERS_CAPS["FP8Linear"] = False

    # Does `from_pretrained` accept the modern `dtype=` kwarg or only `torch_dtype=`?
    import inspect as _inspect
    try:
        _sig = _inspect.signature(transformers.AutoModel.from_pretrained)
        TRANSFORMERS_CAPS["from_pretrained_params"] = sorted(_sig.parameters.keys())
    except Exception:
        TRANSFORMERS_CAPS["from_pretrained_params"] = []

print("=" * 78)
print("TRANSFORMERS CAPABILITIES")
print("=" * 78)
for k, v in TRANSFORMERS_CAPS.items():
    if k != "from_pretrained_params":
        print(f"{k:<38}: {v}")
print("\nNOTE: presence of FP8Linear / FineGrainedFP8Config does NOT mean we should use them.")
print("      Section 9 inspects the checkpoint's own quantization_config and we honour that.")

In [ ]:
# ---- Dependency gate -------------------------------------------------------
# Inspect first, install only what is genuinely missing, and never touch torch/transformers.
AUTO_INSTALL_MISSING = False   # set True ONLY if you accept pip installs in this Domino run

REQUIRED_MODULES = {
    "torch": None,          # must already exist (CUDA build) - never auto-installed
    "transformers": None,   # must already exist - never auto-installed
    "PIL": "Pillow",
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
}
# PyMuPDF is handled separately because of the fitz/pymupdf import-name ambiguity.
NEVER_AUTO_INSTALL = {"torch", "torchvision", "transformers", "accelerate"}

missing = [m for m in REQUIRED_MODULES if importlib.util.find_spec(m) is None]
if PYMUPDF is None:
    missing.append("pymupdf")

hard_blockers = [m for m in missing if m in NEVER_AUTO_INSTALL]
installable = [REQUIRED_MODULES.get(m, m) for m in missing if m not in NEVER_AUTO_INSTALL]

if not missing:
    print("[OK] All required dependencies are present. Nothing to install.")
else:
    print("[MISSING]", missing)
    if hard_blockers:
        raise RuntimeError(
            f"Critical dependencies missing from the Domino image: {hard_blockers}. "
            "Do NOT pip-install these here - use a Domino environment that already ships "
            "a CUDA-enabled torch + transformers."
        )
    cmd = f"{sys.executable} -m pip install --no-input " + " ".join(installable)
    print("Minimal install command:\n   ", cmd)
    if AUTO_INSTALL_MISSING:
        import subprocess
        subprocess.run(cmd, shell=True, check=True)
        print("\n[INSTALLED] Restart the kernel, then re-run from the top.")
    else:
        raise RuntimeError(
            "Missing optional-but-required packages. Either run the command above manually "
            "or set AUTO_INSTALL_MISSING = True and re-run this cell."
        )

## 2. Configuration

Single source of truth for the whole run. Two modes are supported:

* `RUN_MODE = "DIAGNOSTIC"` — deterministic subset (first *N* customers in sorted order), page caps on.
* `RUN_MODE = "FULL"` — every customer, every page, no caps.

Generation is deterministic by default (`do_sample=False`), which is the correct choice for
transcription: we want faithful OCR, not creative completion.

`MAX_NEW_TOKENS = 1024` is a deliberate default. A single KYC page rarely exceeds ~600–800 tokens of
transcription; oversized `max_new_tokens` was the dominant cause of runaway runtime in earlier
implementations (the model rambles or loops until the cap). Raise it only if you observe truncated
output (`finish_reason`-style truncation is reported per page as `hit_token_cap`).

In [ ]:
import random
from pathlib import Path

CONFIG = {
    # ---------------- Run mode ----------------
    "run_mode": "DIAGNOSTIC",              # "DIAGNOSTIC" or "FULL"
    "process_all_customers": False,        # forced True when run_mode == "FULL"
    "max_customers": 5,                    # DIAGNOSTIC only
    "selected_customers": [],              # explicit customer_ids override the subset logic
    "max_pages_per_pdf": None,             # None = every page (set e.g. 2 for a very fast smoke test)
    "max_total_pages": None,               # global page budget; None = unlimited
    "random_seed": 1234,

    # ---------------- Paths ----------------
    "zip_path": None,                      # None = auto-discover "kyc_documents.zip"
    "zip_search_roots": [".", "/mnt", "/mnt/data", "/domino/datasets", "/domino/datasets/local",
                          "/workspace", "/home", str(Path.home())],
    "work_dir": "./kyc_stage1_work",
    "extract_dir": "./kyc_stage1_work/extracted",
    "output_dir": "./outputs",
    "force_reextract": False,              # True = wipe extract dir and unzip again

    # ---------------- Model ----------------
    "model_path": "/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.6-27B-FP8/main",
    "trust_remote_code": True,             # local checkpoint, local code only (no network)
    "device_map": "auto",                  # "auto" | "cuda:0" | None
    "attn_implementation": "auto",         # "auto" -> flash_attention_2 if importable, else sdpa, else eager
    "model_dtype": "auto",                 # "auto" = honour the checkpoint's own dtype/quantization

    # ---------------- Rendering ----------------
    "render_dpi": 200,                     # 200 DPI is the sweet spot for degraded KYC scans
    "max_image_long_side": 1800,           # downscale only; NEVER upscale
    "min_image_long_side": 640,            # below this we warn (page likely unreadable) but do not upscale
    "enable_optional_preprocessing": False,  # keep the baseline RAW (see section 13)

    # ---------------- Generation ----------------
    "max_new_tokens": 1024,
    "temperature": 0.0,
    "top_p": 1.0,
    "do_sample": False,                    # deterministic baseline
    "repetition_penalty": None,            # None = do not pass it at all

    # ---------------- Warmup ----------------
    "warmup_iterations": 2,
    "warmup_max_new_tokens": 32,

    # ---------------- Misc ----------------
    "save_page_images": False,             # True = also dump rendered PNGs (debug; heavy on disk)
    "save_per_page_text_files": True,
    "progress_every": 1,                   # print progress every N pages
}

if CONFIG["run_mode"].upper() == "FULL":
    CONFIG["process_all_customers"] = True
    CONFIG["max_customers"] = None
    CONFIG["max_pages_per_pdf"] = None
    CONFIG["max_total_pages"] = None

# ---- Required documents: canonical names + safe aliases --------------------
# Matching is DETERMINISTIC: a file matches only if its normalized stem is in the
# alias set below. No fuzzy/edit-distance matching -> no silent wrong matches.
# NOTE: "CARTON SIGNATUTE" is the spelling given in the specification (a typo that
# exists in the source data); "CARTON SIGNATURE" is accepted as a safe alias.
REQUIRED_DOCUMENTS = {
    "identity":  {"canonical": "JUSTIFICATIF IDENTITE.PDF",
                  "aliases": ["JUSTIFICATIF IDENTITE", "JUSTIFICATIFIDENTITE",
                              "JUSTIFICATIF D IDENTITE", "JUSTIFICATIF DE IDENTITE",
                              "JUSTIFICATIF D'IDENTITE"]},
    "domicile":  {"canonical": "JUSTIFICATIF DOMICILE.PDF",
                  "aliases": ["JUSTIFICATIF DOMICILE", "JUSTIFICATIFDOMICILE",
                              "JUSTIFICATIF DE DOMICILE"]},
    "convention": {"canonical": "CONVENTION COMPTE.PDF",
                   "aliases": ["CONVENTION COMPTE", "CONVENTIONCOMPTE",
                               "CONVENTION DE COMPTE"]},
    "fatca":     {"canonical": "FATCA.PDF",
                  "aliases": ["FATCA"]},
    "signature": {"canonical": "CARTON SIGNATUTE.PDF",
                  "aliases": ["CARTON SIGNATUTE", "CARTONSIGNATUTE",
                              "CARTON SIGNATURE", "CARTONSIGNATURE",
                              "CARTON DE SIGNATURE"]},
}
DOCUMENT_KEYS = list(REQUIRED_DOCUMENTS.keys())

random.seed(CONFIG["random_seed"])
try:
    import numpy as np
    np.random.seed(CONFIG["random_seed"])
except Exception:
    np = None
if torch is not None:
    torch.manual_seed(CONFIG["random_seed"])
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(CONFIG["random_seed"])

# ---- Output directory layout ----------------------------------------------
OUT = Path(CONFIG["output_dir"]).resolve()
DIRS = {
    "root": OUT,
    "inventory": OUT / "inventory",
    "raw_ocr": OUT / "raw_ocr",
    "raw_ocr_pages": OUT / "raw_ocr" / "pages",
    "raw_ocr_responses": OUT / "raw_ocr" / "raw_model_responses",
    "performance": OUT / "performance",
    "errors": OUT / "errors",
    "diagnostics": OUT / "diagnostics",
    "page_images": OUT / "diagnostics" / "page_images",
}
for _p in DIRS.values():
    _p.mkdir(parents=True, exist_ok=True)
Path(CONFIG["work_dir"]).mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
print("RUN_ID:", RUN_ID)
print(json.dumps({k: (str(v) if isinstance(v, Path) else v) for k, v in CONFIG.items()},
                 indent=2, ensure_ascii=False))
print("\nOutput layout:")
for k, v in DIRS.items():
    print(f"  {k:<20} -> {v}")

## 3. Dataset discovery

Locate `kyc_documents.zip` without modifying it. We inspect the archive's central directory
*before* extracting anything, so structural surprises (nested wrapper folder, loose files at root,
non-PDF payloads, path-traversal entries) are known up front.

In [ ]:
import zipfile

ZIP_NAME = "kyc_documents.zip"


def discover_zip(explicit_path=None, roots=None, max_depth=4):
    """Find the dataset archive. Explicit path wins; otherwise do a bounded search."""
    if explicit_path:
        p = Path(explicit_path).expanduser().resolve()
        if not p.is_file():
            raise FileNotFoundError(f"Configured zip_path does not exist: {p}")
        return p

    seen, candidates = set(), []
    for root in (roots or []):
        root_path = Path(root).expanduser()
        if not root_path.is_dir():
            continue
        root_resolved = str(root_path.resolve())
        if root_resolved in seen:
            continue
        seen.add(root_resolved)
        base_depth = len(root_path.resolve().parts)
        for dirpath, dirnames, filenames in os.walk(root_path, topdown=True):
            depth = len(Path(dirpath).resolve().parts) - base_depth
            if depth >= max_depth:
                dirnames[:] = []
            dirnames[:] = [d for d in dirnames
                           if not d.startswith(".") and d not in
                           {"node_modules", "site-packages", "__pycache__", "proc", "sys", "modelhub"}]
            for fn in filenames:
                if fn.lower() == ZIP_NAME.lower():
                    candidates.append(Path(dirpath) / fn)
        if candidates:
            break
    if not candidates:
        raise FileNotFoundError(
            f"'{ZIP_NAME}' not found under {roots}. Set CONFIG['zip_path'] explicitly."
        )
    candidates.sort(key=lambda p: (len(p.resolve().parts), str(p)))
    return candidates[0].resolve()


ZIP_PATH = discover_zip(CONFIG["zip_path"], CONFIG["zip_search_roots"])
print("ZIP found :", ZIP_PATH)
print("ZIP size  :", f"{ZIP_PATH.stat().st_size / (1024**2):.2f} MiB")

# ---- Inspect the archive WITHOUT extracting --------------------------------
with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    infos = zf.infolist()

zip_entries = [{
    "name": i.filename,
    "is_dir": i.is_dir(),
    "size": i.file_size,
    "compressed": i.compress_size,
} for i in infos]

n_dirs = sum(1 for e in zip_entries if e["is_dir"])
n_files = sum(1 for e in zip_entries if not e["is_dir"])
n_pdfs = sum(1 for e in zip_entries if not e["is_dir"] and e["name"].lower().endswith(".pdf"))
suspicious = [e["name"] for e in zip_entries
              if e["name"].startswith("/") or ".." in Path(e["name"]).parts]

ZIP_INSPECTION = {
    "zip_path": str(ZIP_PATH),
    "entries_total": len(zip_entries),
    "directories": n_dirs,
    "files": n_files,
    "pdf_files": n_pdfs,
    "non_pdf_files": n_files - n_pdfs,
    "suspicious_entries": suspicious,
    "top_level_entries": sorted({Path(e["name"]).parts[0] for e in zip_entries if e["name"].strip()}),
}
print("\nArchive inspection:")
print(json.dumps({k: (v if not isinstance(v, list) or len(v) <= 12 else v[:12] + ["..."])
                  for k, v in ZIP_INSPECTION.items()}, indent=2, ensure_ascii=False))

print("\nFirst 20 entries:")
for e in zip_entries[:20]:
    print(f"  [{'D' if e['is_dir'] else 'F'}] {e['name']}  ({e['size']} bytes)")

if suspicious:
    print("\n[WARNING] Path-traversal-looking entries detected; they will be REJECTED at extraction.")

## 4. ZIP extraction

Safe extraction: every member's resolved destination must stay inside the extraction root
(defence against `../` and absolute-path entries). The original archive is opened read-only and
never modified.

In [ ]:
import shutil
import time

EXTRACT_ROOT = Path(CONFIG["extract_dir"]).resolve()


def safe_extract(zip_path: Path, dest_root: Path, force: bool = False):
    """Extract with path-traversal protection. Returns (stats, rejected_members)."""
    if force and dest_root.exists():
        shutil.rmtree(dest_root)
    dest_root.mkdir(parents=True, exist_ok=True)

    already = any(dest_root.iterdir())
    if already and not force:
        return {"skipped": True, "reason": "destination not empty (set force_reextract=True to redo)",
                "extracted_files": 0, "extracted_dirs": 0, "elapsed_s": 0.0}, []

    rejected, n_files, n_dirs = [], 0, 0
    t0 = time.perf_counter()
    with zipfile.ZipFile(zip_path, "r") as zf:
        for member in zf.infolist():
            name = member.filename
            if not name or name.startswith("/") or ".." in Path(name).parts:
                rejected.append({"name": name, "reason": "path traversal / absolute path"})
                continue
            target = (dest_root / name).resolve()
            if not str(target).startswith(str(dest_root) + os.sep) and target != dest_root:
                rejected.append({"name": name, "reason": "escapes extraction root"})
                continue
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                n_dirs += 1
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(member, "r") as src, open(target, "wb") as dst:
                shutil.copyfileobj(src, dst, length=1024 * 1024)
            n_files += 1
    return {"skipped": False, "extracted_files": n_files, "extracted_dirs": n_dirs,
            "elapsed_s": round(time.perf_counter() - t0, 3)}, rejected


EXTRACTION_STATS, REJECTED_MEMBERS = safe_extract(ZIP_PATH, EXTRACT_ROOT, CONFIG["force_reextract"])
print("Extraction root:", EXTRACT_ROOT)
print(json.dumps(EXTRACTION_STATS, indent=2))
if REJECTED_MEMBERS:
    print("\n[SECURITY] Rejected members:")
    for r in REJECTED_MEMBERS[:20]:
        print("  ", r)

# ---- Locate the real dataset root (handles a single wrapper folder) --------
def resolve_dataset_root(root: Path) -> Path:
    current = root
    for _ in range(4):
        children = [c for c in current.iterdir() if not c.name.startswith(("__MACOSX", "."))]
        dirs = [c for c in children if c.is_dir()]
        files = [c for c in children if c.is_file()]
        if len(dirs) == 1 and not files:
            current = dirs[0]
            continue
        break
    return current


DATASET_ROOT = resolve_dataset_root(EXTRACT_ROOT)
print("\nDataset root:", DATASET_ROOT)
print("Top-level children (first 20):")
for c in sorted(DATASET_ROOT.iterdir())[:20]:
    print(f"  [{'D' if c.is_dir() else 'F'}] {c.name}")

## 5. Customer / document inventory

Customer folders are the directories directly under the dataset root; the folder name **is** the
client identifier. Every file is inventoried and classified as *required-document match* or *other*.

Filename normalization (deterministic, no fuzzy matching):

1. strip the extension, keep only `.pdf`/`.PDF` files as PDF candidates
2. Unicode NFKD + accent removal, uppercase
3. underscores/hyphens/dots -> spaces, whitespace collapsed, trimmed
4. compare against the alias set, and against the alias set with all spaces removed

Anything that does not match exactly is labelled `other` and is **not** processed.

In [ ]:
import re
import unicodedata
from collections import Counter, defaultdict


def normalize_name(name: str) -> str:
    """Deterministic filename normalization: NFKD, de-accent, uppercase, collapse separators."""
    s = unicodedata.normalize("NFKD", str(name))
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = s.upper()
    s = re.sub(r"[_\-.]+", " ", s)
    s = re.sub(r"[^A-Z0-9' ]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def _alias_index():
    """Build {normalized_alias: doc_key} and {despaced_alias: doc_key}."""
    exact, despaced = {}, {}
    for key, spec in REQUIRED_DOCUMENTS.items():
        names = set(spec["aliases"]) | {Path(spec["canonical"]).stem}
        for alias in names:
            n = normalize_name(alias)
            exact[n] = key
            despaced[n.replace(" ", "").replace("'", "")] = key
    return exact, despaced


ALIAS_EXACT, ALIAS_DESPACED = _alias_index()


def classify_file(path: Path):
    """Return (doc_key or None, normalized_stem, is_pdf)."""
    is_pdf = path.suffix.lower() == ".pdf"
    stem_norm = normalize_name(path.stem)
    if not is_pdf:
        return None, stem_norm, False
    key = ALIAS_EXACT.get(stem_norm)
    if key is None:
        key = ALIAS_DESPACED.get(stem_norm.replace(" ", "").replace("'", ""))
    return key, stem_norm, True


# ---- Walk the dataset ------------------------------------------------------
customer_dirs = sorted([d for d in DATASET_ROOT.iterdir()
                        if d.is_dir() and not d.name.startswith(("__MACOSX", "."))],
                       key=lambda p: p.name)

INVENTORY_ROWS = []
STRUCTURE_WARNINGS = []
loose_root_files = [f for f in DATASET_ROOT.iterdir() if f.is_file() and not f.name.startswith(".")]
if loose_root_files:
    STRUCTURE_WARNINGS.append({
        "type": "loose_files_at_dataset_root",
        "count": len(loose_root_files),
        "examples": [f.name for f in loose_root_files[:10]],
    })

for cdir in customer_dirs:
    customer_id = cdir.name
    nested_dirs = [d for d in cdir.iterdir() if d.is_dir() and not d.name.startswith(".")]
    if nested_dirs:
        STRUCTURE_WARNINGS.append({
            "type": "nested_subdirectories_in_customer_folder",
            "customer_id": customer_id,
            "subdirs": [d.name for d in nested_dirs[:10]],
        })
    # rglob so files inside an unexpected sub-folder are still inventoried (and flagged)
    for f in sorted(cdir.rglob("*")):
        if not f.is_file() or f.name.startswith("."):
            continue
        key, stem_norm, is_pdf = classify_file(f)
        try:
            size = f.stat().st_size
        except OSError:
            size = -1
        INVENTORY_ROWS.append({
            "customer_id": customer_id,
            "file_name": f.name,
            "relative_path": str(f.relative_to(DATASET_ROOT)),
            "absolute_path": str(f.resolve()),
            "extension": f.suffix.lower(),
            "is_pdf": is_pdf,
            "normalized_stem": stem_norm,
            "matched_document_key": key or "",
            "matched_document_name": REQUIRED_DOCUMENTS[key]["canonical"] if key else "",
            "is_required_document": bool(key),
            "file_size_bytes": size,
            "in_subdirectory": f.parent != cdir,
            "empty_file": size == 0,
        })

import pandas as pd
inventory_df = pd.DataFrame(INVENTORY_ROWS)

print(f"Customer folders : {len(customer_dirs)}")
print(f"Files inventoried: {len(inventory_df)}")
if len(inventory_df):
    print(f"PDF files        : {int(inventory_df['is_pdf'].sum())}")
    print(f"Required-doc PDFs: {int(inventory_df['is_required_document'].sum())}")
    print(f"Other files      : {int((~inventory_df['is_required_document']).sum())}")
    print(f"Empty (0-byte)   : {int(inventory_df['empty_file'].sum())}")
    print("\nExtension distribution:")
    print(inventory_df["extension"].value_counts().to_string())
    print("\nUnmatched PDF stems (NOT processed) - top 20:")
    unmatched = inventory_df[(inventory_df["is_pdf"]) & (~inventory_df["is_required_document"])]
    print(unmatched["normalized_stem"].value_counts().head(20).to_string() if len(unmatched) else "  (none)")

if STRUCTURE_WARNINGS:
    print(f"\n[STRUCTURE WARNINGS] {len(STRUCTURE_WARNINGS)} (first 5):")
    for w in STRUCTURE_WARNINGS[:5]:
        print("  ", w)

inventory_df.to_csv(DIRS["inventory"] / "file_inventory.csv", index=False, encoding="utf-8-sig")
(DIRS["inventory"] / "structure_warnings.json").write_text(
    json.dumps(STRUCTURE_WARNINGS, indent=2, ensure_ascii=False), encoding="utf-8")
(DIRS["inventory"] / "zip_inspection.json").write_text(
    json.dumps({"zip": ZIP_INSPECTION, "extraction": EXTRACTION_STATS,
                "rejected_members": REJECTED_MEMBERS}, indent=2, ensure_ascii=False), encoding="utf-8")
print("\nSaved:", DIRS["inventory"] / "file_inventory.csv")

## 6. Document existence report

One row per `(customer, required_document)` pair — **five rows per customer, always**, so missing
documents are explicit rather than absent. Duplicate matches for the same document key are flagged
and the deterministic winner is the lexicographically first relative path.

In [ ]:
existence_rows = []
duplicate_flags = []

by_customer = defaultdict(dict)
if len(inventory_df):
    req = inventory_df[inventory_df["is_required_document"]]
    for (cust, key), group in req.groupby(["customer_id", "matched_document_key"]):
        group = group.sort_values("relative_path")
        if len(group) > 1:
            duplicate_flags.append({
                "customer_id": cust, "document_key": key,
                "candidates": group["relative_path"].tolist(),
                "selected": group.iloc[0]["relative_path"],
            })
        by_customer[cust][key] = group.iloc[0]

for cdir in customer_dirs:
    cust = cdir.name
    for key in DOCUMENT_KEYS:
        canonical = REQUIRED_DOCUMENTS[key]["canonical"]
        row = by_customer.get(cust, {}).get(key)
        if row is None:
            existence_rows.append({
                "customer_id": cust,
                "document_key": key,
                "document_name": canonical,
                "exists": False,
                "path": "",
                "relative_path": "",
                "file_size_bytes": 0,
                "number_of_pages": None,
                "readable": None,
                "status": "MISSING",
                "error_message": "",
            })
        else:
            existence_rows.append({
                "customer_id": cust,
                "document_key": key,
                "document_name": canonical,
                "exists": True,
                "path": row["absolute_path"],
                "relative_path": row["relative_path"],
                "file_size_bytes": int(row["file_size_bytes"]),
                "number_of_pages": None,      # filled in section 7
                "readable": None,             # filled in section 7
                "status": "EXISTS_UNVALIDATED",
                "error_message": "",
            })

existence_df = pd.DataFrame(existence_rows)
print(f"Existence rows: {len(existence_df)}  ({len(customer_dirs)} customers x {len(DOCUMENT_KEYS)} documents)")
print("\nPresence per document type:")
print(existence_df.groupby("document_name")["exists"].agg(["sum", "count"]).rename(
    columns={"sum": "present", "count": "customers"}).to_string())
if duplicate_flags:
    print(f"\n[DUPLICATES] {len(duplicate_flags)} customer/document pairs had multiple candidates:")
    for d in duplicate_flags[:5]:
        print("  ", d)

## 7. PDF readability validation

Every existing PDF is opened with PyMuPDF to determine (a) whether it is genuinely readable and
(b) its **actual, dynamic** page count. Nothing about page numbering or page counts is assumed
anywhere downstream. Encrypted PDFs are detected and an empty-password unlock is attempted once.

Final status vocabulary: `MISSING` · `EXISTS_UNREADABLE` · `EXISTS_READABLE_EMPTY` · `EXISTS_READABLE`.

In [ ]:
import traceback

if PYMUPDF is None:
    raise RuntimeError("PyMuPDF is required for PDF validation/rendering (see section 1).")


def probe_pdf(path: str):
    """Open a PDF defensively. Returns (readable, n_pages, error_message, meta)."""
    doc = None
    try:
        doc = PYMUPDF.open(path)
        if getattr(doc, "needs_pass", False):
            if not doc.authenticate(""):
                return False, None, "PDF is encrypted and requires a password", {}
        n_pages = doc.page_count
        meta = {"is_pdf": bool(getattr(doc, "is_pdf", True)),
                "encrypted": bool(getattr(doc, "is_encrypted", False)),
                "metadata_title": (doc.metadata or {}).get("title", "")}
        if n_pages > 0:
            _ = doc.load_page(0)  # touch page 0 to catch lazily-raised corruption
        return True, int(n_pages), "", meta
    except Exception as exc:
        return False, None, f"{type(exc).__name__}: {exc}", {}
    finally:
        try:
            if doc is not None:
                doc.close()
        except Exception:
            pass


validation_t0 = time.perf_counter()
for idx, row in existence_df.iterrows():
    if not row["exists"]:
        continue
    readable, n_pages, err, meta = probe_pdf(row["path"])
    existence_df.at[idx, "readable"] = bool(readable)
    existence_df.at[idx, "number_of_pages"] = n_pages
    existence_df.at[idx, "error_message"] = err
    if not readable:
        existence_df.at[idx, "status"] = "EXISTS_UNREADABLE"
    elif not n_pages:
        existence_df.at[idx, "status"] = "EXISTS_READABLE_EMPTY"
    else:
        existence_df.at[idx, "status"] = "EXISTS_READABLE"
validation_elapsed = time.perf_counter() - validation_t0

print(f"Validated {int(existence_df['exists'].sum())} PDFs in {validation_elapsed:.2f}s")
print("\nStatus distribution:")
print(existence_df["status"].value_counts().to_string())
_pages = pd.to_numeric(existence_df["number_of_pages"], errors="coerce")
print(f"\nTotal pages across readable PDFs: {int(_pages.fillna(0).sum())}")
if _pages.notna().any():
    print("Pages per document -> min={:.0f} median={:.0f} max={:.0f}".format(
        _pages.min(), _pages.median(), _pages.max()))

# ---- Customer-level summary ------------------------------------------------
summary_rows = []
for cust, grp in existence_df.groupby("customer_id"):
    present = {k: bool(grp[(grp["document_key"] == k) & (grp["exists"])].shape[0]) for k in DOCUMENT_KEYS}
    readable_map = {k: bool(grp[(grp["document_key"] == k) & (grp["status"] == "EXISTS_READABLE")].shape[0])
                    for k in DOCUMENT_KEYS}
    pages = pd.to_numeric(grp["number_of_pages"], errors="coerce").fillna(0).sum()
    summary_rows.append({
        "customer_id": cust,
        "identity_exists": present["identity"],
        "domicile_exists": present["domicile"],
        "convention_exists": present["convention"],
        "fatca_exists": present["fatca"],
        "signature_exists": present["signature"],
        "number_of_required_documents_present": int(sum(present.values())),
        "number_of_required_documents_readable": int(sum(readable_map.values())),
        "total_pages_available": int(pages),
        "complete_folder": bool(sum(present.values()) == len(DOCUMENT_KEYS)),
    })
summary_df = pd.DataFrame(summary_rows).sort_values("customer_id").reset_index(drop=True)

print("\nCustomer completeness (documents present per folder):")
print(summary_df["number_of_required_documents_present"].value_counts().sort_index().to_string())
print(f"Complete folders (5/5): {int(summary_df['complete_folder'].sum())} / {len(summary_df)}")

# ---- Persist reports (CSV + JSON) -----------------------------------------
existence_df.to_csv(DIRS["inventory"] / "document_existence_report.csv", index=False, encoding="utf-8-sig")
existence_df.to_json(DIRS["inventory"] / "document_existence_report.json",
                     orient="records", indent=2, force_ascii=False)
summary_df.to_csv(DIRS["inventory"] / "customer_summary.csv", index=False, encoding="utf-8-sig")
summary_df.to_json(DIRS["inventory"] / "customer_summary.json",
                   orient="records", indent=2, force_ascii=False)
(DIRS["inventory"] / "duplicate_document_flags.json").write_text(
    json.dumps(duplicate_flags, indent=2, ensure_ascii=False), encoding="utf-8")

print("\nSaved reports to", DIRS["inventory"])
display(existence_df.head(12))
display(summary_df.head(12))

## 8. GPU diagnostics

Helper used consistently before/after model load, around warmup and per page. All numbers come
from the PyTorch CUDA caching allocator (`allocated` / `reserved` / `max_*`) plus the driver's
`mem_get_info` for true free/total VRAM.

We **do not** call `torch.cuda.empty_cache()` per page — that forces the allocator to release and
re-acquire segments and measurably slows steady-state inference. It is called exactly once, before
model loading, to start from a clean baseline.

In [ ]:
def gpu_memory_snapshot(device: int = 0, label: str = ""):
    """Return a dict of CUDA allocator + driver memory stats (bytes and GiB)."""
    if torch is None or not torch.cuda.is_available():
        return {"label": label, "cuda": False}
    free_b, total_b = torch.cuda.mem_get_info(device)
    snap = {
        "label": label,
        "cuda": True,
        "device": device,
        "device_name": torch.cuda.get_device_name(device),
        "allocated_bytes": torch.cuda.memory_allocated(device),
        "reserved_bytes": torch.cuda.memory_reserved(device),
        "max_allocated_bytes": torch.cuda.max_memory_allocated(device),
        "max_reserved_bytes": torch.cuda.max_memory_reserved(device),
        "free_bytes": free_b,
        "total_bytes": total_b,
    }
    for k in list(snap):
        if k.endswith("_bytes"):
            snap[k.replace("_bytes", "_gib")] = round(snap[k] / (1024 ** 3), 3)
    return snap


def print_gpu_snapshot(snap):
    if not snap.get("cuda"):
        print(f"[{snap.get('label','')}] CUDA not available")
        return
    print(f"[{snap['label']}] {snap['device_name']}  "
          f"allocated={snap['allocated_gib']:.2f} GiB  reserved={snap['reserved_gib']:.2f} GiB  "
          f"peak_alloc={snap['max_allocated_gib']:.2f} GiB  peak_reserved={snap['max_reserved_gib']:.2f} GiB  "
          f"free={snap['free_gib']:.2f}/{snap['total_gib']:.2f} GiB")


def cuda_sync():
    """Synchronize only at timing boundaries that actually matter (H2D, generate)."""
    if torch is not None and torch.cuda.is_available():
        torch.cuda.synchronize()


if torch is not None and torch.cuda.is_available():
    torch.cuda.empty_cache()                 # once, before loading - not per page
    torch.cuda.reset_peak_memory_stats()

GPU_BEFORE_LOAD = gpu_memory_snapshot(label="before_model_load")
print_gpu_snapshot(GPU_BEFORE_LOAD)

## 9. Qwen3.6 model inspection (no loading yet)

We read the checkpoint's own files to learn what it *is* before deciding how to load it:
architecture(s), model type, dtype, `quantization_config`, vision/image-processor settings, context
length, and whether a chat template ships with the processor.

Explicitly: we do **not** assume `FP8Linear` / `FineGrainedFP8Config` apply here. If the checkpoint
declares its own `quantization_config`, `from_pretrained` honours it automatically and we pass no
quantization argument at all.

In [ ]:
MODEL_PATH = Path(CONFIG["model_path"])

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Model path does not exist: {MODEL_PATH}\n"
        "Check the Domino ModelHub mount. This notebook never downloads from the Hugging Face Hub."
    )

# Work fully offline: any accidental hub call fails fast instead of hanging.
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")


def _read_json(p: Path):
    try:
        return json.loads(p.read_text(encoding="utf-8"))
    except Exception as exc:
        return {"__error__": f"{type(exc).__name__}: {exc}"}


print("Model directory:", MODEL_PATH)
files = sorted(MODEL_PATH.iterdir(), key=lambda p: p.name)
total_bytes = 0
print(f"\n{len(files)} entries:")
for f in files:
    if f.is_file():
        sz = f.stat().st_size
        total_bytes += sz
        print(f"  {f.name:<52} {sz / (1024**2):>10.1f} MiB")
    else:
        print(f"  {f.name + '/':<52} {'<dir>':>10}")
print(f"\nTotal checkpoint size: {total_bytes / (1024**3):.2f} GiB")

MODEL_CONFIG_RAW = _read_json(MODEL_PATH / "config.json") if (MODEL_PATH / "config.json").exists() else {}
PREPROCESSOR_RAW = _read_json(MODEL_PATH / "preprocessor_config.json") if (MODEL_PATH / "preprocessor_config.json").exists() else {}
GENERATION_RAW = _read_json(MODEL_PATH / "generation_config.json") if (MODEL_PATH / "generation_config.json").exists() else {}
TOKENIZER_RAW = _read_json(MODEL_PATH / "tokenizer_config.json") if (MODEL_PATH / "tokenizer_config.json").exists() else {}
PROCESSOR_RAW = _read_json(MODEL_PATH / "processor_config.json") if (MODEL_PATH / "processor_config.json").exists() else {}

MODEL_INSPECTION = {
    "model_path": str(MODEL_PATH),
    "checkpoint_size_gib": round(total_bytes / (1024 ** 3), 2),
    "architectures": MODEL_CONFIG_RAW.get("architectures"),
    "model_type": MODEL_CONFIG_RAW.get("model_type"),
    "torch_dtype": MODEL_CONFIG_RAW.get("torch_dtype") or MODEL_CONFIG_RAW.get("dtype"),
    "quantization_config": MODEL_CONFIG_RAW.get("quantization_config"),
    "max_position_embeddings": (MODEL_CONFIG_RAW.get("max_position_embeddings")
                                or (MODEL_CONFIG_RAW.get("text_config") or {}).get("max_position_embeddings")),
    "vision_config_keys": sorted((MODEL_CONFIG_RAW.get("vision_config") or {}).keys()),
    "text_config_keys": sorted((MODEL_CONFIG_RAW.get("text_config") or {}).keys()),
    "image_processor_type": PREPROCESSOR_RAW.get("image_processor_type"),
    "processor_class": MODEL_CONFIG_RAW.get("processor_class") or PREPROCESSOR_RAW.get("processor_class"),
    "min_pixels": PREPROCESSOR_RAW.get("min_pixels"),
    "max_pixels": PREPROCESSOR_RAW.get("max_pixels"),
    "patch_size": PREPROCESSOR_RAW.get("patch_size"),
    "merge_size": PREPROCESSOR_RAW.get("merge_size"),
    "chat_template_in_tokenizer_config": "chat_template" in TOKENIZER_RAW,
    "chat_template_file_present": (MODEL_PATH / "chat_template.json").exists() or (MODEL_PATH / "chat_template.jinja").exists(),
    "generation_config": GENERATION_RAW,
    "has_safetensors": any(f.suffix == ".safetensors" for f in files if f.is_file()),
    "has_remote_code_files": any(f.name.startswith("modeling_") or f.name.startswith("processing_")
                                 for f in files if f.is_file()),
}

print("\n" + "=" * 78)
print("MODEL INSPECTION")
print("=" * 78)
print(json.dumps(MODEL_INSPECTION, indent=2, ensure_ascii=False)[:6000])

if MODEL_INSPECTION["quantization_config"]:
    print("\n[INFO] The checkpoint declares its own quantization_config (shown above).")
    print("       We pass NO quantization argument to from_pretrained: the checkpoint wins.")
else:
    print("\n[INFO] No quantization_config in config.json - loading with the checkpoint's native dtype.")

(DIRS["diagnostics"] / "model_inspection.json").write_text(
    json.dumps({"inspection": MODEL_INSPECTION, "config_json": MODEL_CONFIG_RAW,
                "preprocessor_config": PREPROCESSOR_RAW, "generation_config": GENERATION_RAW},
               indent=2, ensure_ascii=False), encoding="utf-8")

In [ ]:
# ---- Resolve the AutoConfig and pick the right auto-class ------------------
from transformers import AutoConfig

HF_CONFIG = None
CONFIG_LOAD_ERROR = None
try:
    HF_CONFIG = AutoConfig.from_pretrained(str(MODEL_PATH),
                                           trust_remote_code=CONFIG["trust_remote_code"],
                                           local_files_only=True)
    print("AutoConfig class :", type(HF_CONFIG).__name__)
    print("model_type       :", getattr(HF_CONFIG, "model_type", None))
    print("architectures    :", getattr(HF_CONFIG, "architectures", None))
except Exception as exc:
    CONFIG_LOAD_ERROR = f"{type(exc).__name__}: {exc}"
    print("[WARN] AutoConfig failed:", CONFIG_LOAD_ERROR)
    print("       Falling back to the raw config.json contents for class selection.")

_arch_list = (getattr(HF_CONFIG, "architectures", None) or MODEL_INSPECTION.get("architectures") or [])
ARCH_NAME = _arch_list[0] if _arch_list else ""
print("\nPrimary architecture:", ARCH_NAME or "<unknown>")

IS_MULTIMODAL = bool(
    (MODEL_CONFIG_RAW.get("vision_config"))
    or PREPROCESSOR_RAW.get("image_processor_type")
    or "VL" in ARCH_NAME.upper()
    or "VISION" in ARCH_NAME.upper()
    or "IMAGE" in ARCH_NAME.upper()
)
print("Detected as multimodal (vision-language):", IS_MULTIMODAL)
if not IS_MULTIMODAL:
    print("[WARNING] No vision configuration detected. OCR requires a vision-language checkpoint;")
    print("          verify the ModelHub path points at the multimodal Qwen3.6 build.")


def resolve_model_class(arch_name: str):
    """Pick the loading class: exact architecture class first, then auto-classes by capability."""
    import transformers
    tried = []
    if arch_name and hasattr(transformers, arch_name):
        return getattr(transformers, arch_name), arch_name, tried
    tried.append(f"transformers.{arch_name} (not found)")
    order = (["AutoModelForImageTextToText", "AutoModelForVision2Seq", "AutoModelForCausalLM"]
             if IS_MULTIMODAL else ["AutoModelForCausalLM", "AutoModel"])
    for name in order:
        if hasattr(transformers, name):
            return getattr(transformers, name), name, tried
        tried.append(f"transformers.{name} (not found)")
    raise RuntimeError(f"No suitable model class found. Tried: {tried}")


MODEL_CLASS, MODEL_CLASS_NAME, _tried = resolve_model_class(ARCH_NAME)
print("\nSelected model class:", MODEL_CLASS_NAME, f"({MODEL_CLASS.__name__})")
if _tried:
    print("Skipped:", _tried)


# ---- Attention implementation ---------------------------------------------
def resolve_attn_implementation(preference: str):
    """flash_attention_2 only if flash_attn actually imports; otherwise sdpa; eager as last resort."""
    if preference and preference != "auto":
        return preference, f"explicitly configured: {preference}"
    if importlib.util.find_spec("flash_attn") is not None:
        try:
            importlib.import_module("flash_attn")
            return "flash_attention_2", "flash_attn is installed and importable"
        except Exception as exc:
            return "sdpa", f"flash_attn present but not importable ({type(exc).__name__}); using sdpa"
    if torch is not None and hasattr(torch.nn.functional, "scaled_dot_product_attention"):
        return "sdpa", "flash_attn absent; PyTorch SDPA available (good H100 default)"
    return "eager", "no SDPA available"


ATTN_IMPL, ATTN_REASON = resolve_attn_implementation(CONFIG["attn_implementation"])
print(f"\nAttention implementation: {ATTN_IMPL}  ({ATTN_REASON})")
print("FlashAttention is NOT installed by this notebook. SDPA is a perfectly good H100 fallback.")

## 10. Qwen3.6 model loading

Loaded **once** and reused for the whole run. Key decisions:

* `dtype="auto"` → honour the checkpoint (`torch_dtype` / `quantization_config`). No conversion to
  another quantization format.
* `device_map="auto"` → let accelerate place the shards; on a single H100 this is one device.
* `attn_implementation` resolved above, with an automatic retry on `sdpa` then `eager` if the
  requested backend is rejected by the model class.
* `dtype=` vs legacy `torch_dtype=` is selected from the actual `from_pretrained` signature.

In [ ]:
import inspect

def _dtype_kwarg_name():
    """Newer transformers renamed torch_dtype -> dtype; support both without guessing."""
    try:
        params = set(inspect.signature(MODEL_CLASS.from_pretrained).parameters)
    except (TypeError, ValueError):
        params = set()
    if "dtype" in params:
        return "dtype"
    return "torch_dtype"


DTYPE_KWARG = _dtype_kwarg_name()
print("dtype kwarg used:", DTYPE_KWARG)

base_kwargs = {
    "pretrained_model_name_or_path": str(MODEL_PATH),
    "local_files_only": True,
    "trust_remote_code": CONFIG["trust_remote_code"],
    "low_cpu_mem_usage": True,
    DTYPE_KWARG: CONFIG["model_dtype"],          # "auto" -> checkpoint decides
}
if CONFIG["device_map"]:
    base_kwargs["device_map"] = CONFIG["device_map"]

MODEL = None
MODEL_LOAD_REPORT = {"attempts": []}
t_load0 = time.perf_counter()

for attn in [ATTN_IMPL] + [a for a in ("sdpa", "eager") if a != ATTN_IMPL]:
    kwargs = dict(base_kwargs)
    kwargs["attn_implementation"] = attn
    try:
        print(f"\nLoading with attn_implementation={attn} ...")
        MODEL = MODEL_CLASS.from_pretrained(**kwargs)
        ATTN_IMPL = attn
        MODEL_LOAD_REPORT["attempts"].append({"attn": attn, "ok": True})
        break
    except Exception as exc:
        msg = f"{type(exc).__name__}: {exc}"
        print(f"  [FAILED] {msg.splitlines()[0][:300]}")
        MODEL_LOAD_REPORT["attempts"].append({"attn": attn, "ok": False, "error": msg[:1000]})

if MODEL is None:
    raise RuntimeError("Model loading failed for every attention backend. "
                       f"Details: {json.dumps(MODEL_LOAD_REPORT, indent=2)[:4000]}")

MODEL.eval()
if hasattr(MODEL, "generation_config") and MODEL.generation_config is not None:
    # Deterministic baseline: strip sampling defaults that would otherwise emit warnings.
    MODEL.generation_config.do_sample = CONFIG["do_sample"]
MODEL_LOAD_TIME_S = time.perf_counter() - t_load0

try:
    MODEL_DEVICE = next(MODEL.parameters()).device
except StopIteration:
    MODEL_DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
MODEL_DTYPE = getattr(MODEL, "dtype", None)

_n_params = None
try:
    _n_params = sum(p.numel() for p in MODEL.parameters())
except Exception:
    pass

MODEL_LOAD_REPORT.update({
    "model_class": type(MODEL).__name__,
    "load_time_s": round(MODEL_LOAD_TIME_S, 3),
    "device": str(MODEL_DEVICE),
    "dtype": str(MODEL_DTYPE),
    "attn_implementation": ATTN_IMPL,
    "parameter_count": _n_params,
    "hf_device_map": getattr(MODEL, "hf_device_map", None),
})

print("\n" + "=" * 78)
print(f"MODEL LOADED in {MODEL_LOAD_TIME_S:.1f}s")
print("=" * 78)
print(json.dumps({k: str(v) for k, v in MODEL_LOAD_REPORT.items() if k != "attempts"},
                 indent=2, ensure_ascii=False))

GPU_AFTER_LOAD = gpu_memory_snapshot(label="after_model_load")
print()
print_gpu_snapshot(GPU_BEFORE_LOAD)
print_gpu_snapshot(GPU_AFTER_LOAD)
if GPU_AFTER_LOAD.get("cuda"):
    print(f"\nWeights footprint (delta allocated): "
          f"{GPU_AFTER_LOAD['allocated_gib'] - GPU_BEFORE_LOAD['allocated_gib']:.2f} GiB")

## 11. Processor initialization

The processor is loaded from the same local directory and then **probed** to discover which calling
convention this transformers/checkpoint combination supports. Three strategies, tried in order on a
tiny synthetic image:

1. `chat_template_dict` — `processor.apply_chat_template(..., tokenize=True, return_dict=True)`
   (modern path; handles the image itself)
2. `text_plus_images` — `apply_chat_template(..., tokenize=False)` then `processor(text=..., images=...)`
   (classic Qwen-VL path; uses `qwen_vl_utils.process_vision_info` when available)
3. `plain_text_images` — no chat template available: pass the bare prompt plus the image

The winning strategy is cached in `PROCESSOR_STRATEGY` and used for every page.

In [ ]:
from transformers import AutoProcessor
from PIL import Image, ImageDraw

t_proc0 = time.perf_counter()

_proc_kwargs = {"trust_remote_code": CONFIG["trust_remote_code"], "local_files_only": True}
# min_pixels/max_pixels are Qwen-VL image-processor knobs; only pass them if accepted.
PROCESSOR = None
try:
    PROCESSOR = AutoProcessor.from_pretrained(str(MODEL_PATH), **_proc_kwargs)
except TypeError as exc:
    print("[WARN] AutoProcessor rejected kwargs, retrying minimal:", exc)
    PROCESSOR = AutoProcessor.from_pretrained(str(MODEL_PATH))
PROCESSOR_INIT_TIME_S = time.perf_counter() - t_proc0

TOKENIZER = getattr(PROCESSOR, "tokenizer", None)
IMAGE_PROCESSOR = getattr(PROCESSOR, "image_processor", None)

print(f"Processor: {type(PROCESSOR).__name__}  (init {PROCESSOR_INIT_TIME_S:.2f}s)")
print("Tokenizer:", type(TOKENIZER).__name__ if TOKENIZER is not None else "None")
print("Image processor:", type(IMAGE_PROCESSOR).__name__ if IMAGE_PROCESSOR is not None else "None")
for attr in ("min_pixels", "max_pixels", "patch_size", "merge_size", "size", "do_resize", "do_rescale"):
    if IMAGE_PROCESSOR is not None and hasattr(IMAGE_PROCESSOR, attr):
        print(f"  image_processor.{attr} = {getattr(IMAGE_PROCESSOR, attr)}")

HAS_CHAT_TEMPLATE = bool(getattr(PROCESSOR, "chat_template", None)
                         or (TOKENIZER is not None and getattr(TOKENIZER, "chat_template", None)))
print("\nChat template available:", HAS_CHAT_TEMPLATE)

# Optional Qwen helper for message-embedded images
try:
    from qwen_vl_utils import process_vision_info as _process_vision_info
    HAS_QWEN_VL_UTILS = True
except Exception:
    _process_vision_info = None
    HAS_QWEN_VL_UTILS = False
print("qwen_vl_utils available:", HAS_QWEN_VL_UTILS)

In [ ]:
def build_messages(image, prompt_text):
    """Single-image + instruction user turn (the only message shape this stage uses)."""
    return [{"role": "user",
             "content": [{"type": "image", "image": image},
                         {"type": "text", "text": prompt_text}]}]


def _inputs_look_multimodal(enc):
    keys = set(getattr(enc, "keys", lambda: [])())
    return "input_ids" in keys and any(k in keys for k in
                                       ("pixel_values", "pixel_values_videos", "image_grid_thw", "images"))


def _try_chat_template_dict(image, prompt_text):
    messages = build_messages(image, prompt_text)
    enc = PROCESSOR.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    )
    if not _inputs_look_multimodal(enc):
        raise RuntimeError("apply_chat_template(return_dict=True) produced no image tensors")
    return enc


def _try_text_plus_images(image, prompt_text):
    messages = build_messages(image, prompt_text)
    text = PROCESSOR.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    images = [image]
    if HAS_QWEN_VL_UTILS:
        try:
            img_in, vid_in = _process_vision_info(messages)
            if img_in:
                images = img_in
        except Exception:
            pass
    enc = PROCESSOR(text=[text], images=images, return_tensors="pt", padding=True)
    if not _inputs_look_multimodal(enc):
        raise RuntimeError("processor(text=..., images=...) produced no image tensors")
    return enc


def _try_plain_text_images(image, prompt_text):
    enc = PROCESSOR(text=[prompt_text], images=[image], return_tensors="pt", padding=True)
    if not _inputs_look_multimodal(enc):
        raise RuntimeError("bare processor(text=..., images=...) produced no image tensors")
    return enc


_STRATEGIES = [
    ("chat_template_dict", _try_chat_template_dict),
    ("text_plus_images", _try_text_plus_images),
    ("plain_text_images", _try_plain_text_images),
]


def _probe_image(w=448, h=448):
    img = Image.new("RGB", (w, h), "white")
    d = ImageDraw.Draw(img)
    d.rectangle([24, 24, w - 24, h - 24], outline="black", width=3)
    d.text((48, 64), "CARTE NATIONALE D'IDENTITE", fill="black")
    d.text((48, 96), "Nom / Name: TEST 12345", fill="black")
    d.line([48, 200, w - 48, 200], fill="black", width=2)
    return img


PROBE_IMAGE = _probe_image()
PROCESSOR_STRATEGY = None
STRATEGY_PROBE_LOG = []

for name, fn in _STRATEGIES:
    try:
        _enc = fn(PROBE_IMAGE, "Transcribe this image.")
        PROCESSOR_STRATEGY = name
        STRATEGY_PROBE_LOG.append({"strategy": name, "ok": True,
                                   "keys": sorted(list(_enc.keys()))})
        del _enc
        break
    except Exception as exc:
        STRATEGY_PROBE_LOG.append({"strategy": name, "ok": False,
                                   "error": f"{type(exc).__name__}: {exc}"[:500]})

print("Strategy probe:")
for entry in STRATEGY_PROBE_LOG:
    print("  ", entry)
if PROCESSOR_STRATEGY is None:
    raise RuntimeError("No working processor calling convention found. "
                       f"Probe log: {json.dumps(STRATEGY_PROBE_LOG, indent=2)[:3000]}")
print("\nSelected processor strategy:", PROCESSOR_STRATEGY)

_STRATEGY_FN = dict(_STRATEGIES)[PROCESSOR_STRATEGY]


def build_inputs(image, prompt_text):
    """Encode one (image, prompt) pair using the probed strategy."""
    return _STRATEGY_FN(image, prompt_text)


def move_to_device(enc):
    """Move encodings to the model device; cast pixel values only for fp16/bf16 models."""
    target = MODEL_DEVICE
    try:
        enc = enc.to(target)
    except Exception:
        enc = {k: (v.to(target) if hasattr(v, "to") else v) for k, v in enc.items()}
    if MODEL_DTYPE in (getattr(torch, "float16", None), getattr(torch, "bfloat16", None)):
        for key in ("pixel_values", "pixel_values_videos"):
            v = enc.get(key, None) if hasattr(enc, "get") else None
            if v is not None and hasattr(v, "dtype") and v.dtype.is_floating_point and v.dtype != MODEL_DTYPE:
                enc[key] = v.to(MODEL_DTYPE)
    return enc


def decode_tokens(token_ids, skip_special_tokens=True):
    decoder = PROCESSOR if hasattr(PROCESSOR, "batch_decode") else TOKENIZER
    return decoder.batch_decode(token_ids, skip_special_tokens=skip_special_tokens,
                                clean_up_tokenization_spaces=False)[0]


print("Encoder/decoder helpers ready.")

## 12. Warmup

The first multimodal `generate()` call pays for CUDA context creation, kernel autotuning, lazy
module materialization and allocator growth — it is routinely 5–20x slower than steady state.
Warmup runs on a **synthetic** image with a tiny token cap, and its timings are stored in a
**separate** structure that never enters the measured statistics.

In [ ]:
def generation_kwargs(max_new_tokens=None):
    """Conservative, transcription-oriented generation settings."""
    kw = {
        "max_new_tokens": int(max_new_tokens or CONFIG["max_new_tokens"]),
        "do_sample": bool(CONFIG["do_sample"]),
        "use_cache": True,
    }
    if CONFIG["do_sample"]:
        kw["temperature"] = float(CONFIG["temperature"])
        kw["top_p"] = float(CONFIG["top_p"])
    if CONFIG["repetition_penalty"]:
        kw["repetition_penalty"] = float(CONFIG["repetition_penalty"])
    if TOKENIZER is not None and getattr(TOKENIZER, "pad_token_id", None) is not None:
        kw["pad_token_id"] = TOKENIZER.pad_token_id
    return kw


WARMUP_PROMPT = "Transcribe every visible character in this image. Output only the transcription."
WARMUP_RESULTS = []

if torch is not None and torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

for i in range(int(CONFIG["warmup_iterations"])):
    t0 = time.perf_counter()
    enc = build_inputs(PROBE_IMAGE, WARMUP_PROMPT)
    enc = move_to_device(enc)
    cuda_sync()
    t_prep = time.perf_counter() - t0

    t1 = time.perf_counter()
    with torch.inference_mode():
        out = MODEL.generate(**enc, **generation_kwargs(CONFIG["warmup_max_new_tokens"]))
    cuda_sync()
    t_gen = time.perf_counter() - t1

    in_len = int(enc["input_ids"].shape[-1])
    gen_len = int(out.shape[-1]) - in_len
    text = decode_tokens(out[:, in_len:])
    WARMUP_RESULTS.append({
        "iteration": i + 1,
        "prep_s": round(t_prep, 4),
        "generate_s": round(t_gen, 4),
        "input_tokens": in_len,
        "generated_tokens": gen_len,
        "sample_output": text[:160],
    })
    print(f"warmup {i+1}/{CONFIG['warmup_iterations']}: prep={t_prep:.3f}s generate={t_gen:.3f}s "
          f"in={in_len} out={gen_len} tok")
    del enc, out

WARMUP_TOTAL_S = sum(w["prep_s"] + w["generate_s"] for w in WARMUP_RESULTS)
GPU_AFTER_WARMUP = gpu_memory_snapshot(label="after_warmup")
print(f"\nTotal warmup time: {WARMUP_TOTAL_S:.2f}s  (EXCLUDED from measured statistics)")
print("Last warmup output sample:", WARMUP_RESULTS[-1]["sample_output"] if WARMUP_RESULTS else "(none)")
print()
print_gpu_snapshot(GPU_AFTER_WARMUP)

# Reset peaks so per-page peak memory reflects *inference*, not loading/warmup.
if torch is not None and torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

## 13. PDF page rendering

`PDF page -> PIL image` only. At 200 DPI an A4 page is ~1654×2339 px, which is downscaled to
`max_image_long_side` (1800 by default) — enough to keep small handwriting and stamp text legible
while keeping the visual-token count (and therefore latency) sane.

Rules enforced here:

* aspect ratio preserved, **never upscaled** (upscaling a bad scan adds no information, only tokens)
* no deskew / denoise / sharpen / threshold / crop / rotate in the baseline path
* page rotation flags are respected by PyMuPDF's own rendering; we never re-order or re-orient pages
* grayscale scans are rendered into RGB because vision encoders expect 3 channels — this is a format
  conversion, not preprocessing

`OPTIONAL_PREPROCESSING` below is deliberately isolated and **off** by default. It exists so that a
later experiment can be compared against this baseline, not to improve it silently.

In [ ]:
from PIL import Image  # re-imported here so this section is self-contained

def render_page_to_image(page, dpi: int = None, max_long_side: int = None):
    """Render one PyMuPDF page to a PIL RGB image. Returns (image, render_meta)."""
    dpi = int(dpi or CONFIG["render_dpi"])
    max_long_side = int(max_long_side or CONFIG["max_image_long_side"])
    zoom = dpi / 72.0
    matrix = PYMUPDF.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=matrix, alpha=False, colorspace=PYMUPDF.csRGB)
    img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)

    meta = {
        "dpi": dpi,
        "rendered_width": pix.width,
        "rendered_height": pix.height,
        "page_rotation": int(getattr(page, "rotation", 0) or 0),
        "resized": False,
        "final_width": pix.width,
        "final_height": pix.height,
        "downscale_factor": 1.0,
    }

    long_side = max(img.width, img.height)
    if long_side > max_long_side:                      # downscale only
        scale = max_long_side / float(long_side)
        new_size = (max(1, int(round(img.width * scale))), max(1, int(round(img.height * scale))))
        img = img.resize(new_size, Image.LANCZOS)
        meta.update({"resized": True, "final_width": img.width, "final_height": img.height,
                     "downscale_factor": round(scale, 4)})
    meta["below_min_long_side"] = max(img.width, img.height) < int(CONFIG["min_image_long_side"])
    return img, meta


# ---------------------------------------------------------------------------
# OPTIONAL experimental preprocessing - OFF by default, kept strictly separate
# from the baseline path so that any future A/B is explicit and measurable.
# ---------------------------------------------------------------------------
def OPTIONAL_PREPROCESSING(img):
    """Not part of the raw baseline. Enable via CONFIG['enable_optional_preprocessing']."""
    from PIL import ImageOps
    out = ImageOps.autocontrast(img.convert("L"), cutoff=1).convert("RGB")
    return out


def maybe_preprocess(img):
    """Returns (image, applied_flag). Baseline returns the image untouched."""
    if not CONFIG["enable_optional_preprocessing"]:
        return img, False
    return OPTIONAL_PREPROCESSING(img), True


# ---- Smoke test on the first readable PDF we can find ----------------------
_readable = existence_df[existence_df["status"] == "EXISTS_READABLE"]
if len(_readable):
    _row = _readable.iloc[0]
    _doc = PYMUPDF.open(_row["path"])
    _page = _doc.load_page(0)
    _t0 = time.perf_counter()
    _img, _meta = render_page_to_image(_page)
    _elapsed = time.perf_counter() - _t0
    _doc.close()
    print(f"Render smoke test: {_row['customer_id']} / {_row['document_name']} page 1")
    print(f"  {_elapsed*1000:.0f} ms -> {json.dumps(_meta)}")
    _img.save(DIRS["diagnostics"] / "render_smoke_test.png")
    display(_img.resize((_img.width // 3, _img.height // 3)))
else:
    print("[WARN] No readable PDF available for the render smoke test.")

## 14. Raw OCR inference — prompt and single-page call

### The prompt (section 19 of the specification)

One page = one inference call. Never a whole PDF, never batched pages: a clean per-page baseline is
the entire point of this stage.

The prompt is written to suppress the two failure modes that ruin OCR baselines — *translation* and
*plausible completion*. It forbids translating, transliterating, normalizing, correcting, inferring
and completing, and it asks for `[ILLEGIBLE]` instead of a guess.

In [ ]:
OCR_SYSTEM_PROMPT = (
    "You are a precise document transcription engine. You transcribe scanned documents exactly as "
    "they appear. You never translate, never interpret, and never invent content."
)

OCR_PROMPT = (
    "You are transcribing a scanned document page. Read ONLY what is visibly present in the image.\n"
    "\n"
    "TRANSCRIBE:\n"
    "- every visible text element, in the order it appears on the page\n"
    "- printed text and handwritten text\n"
    "- text in the original language and the original script (French, Arabic, English, mixed)\n"
    "- names exactly as written, in Latin script and/or Arabic script\n"
    "- all numbers, dates, identifiers, account numbers and reference codes\n"
    "- all accents, diacritics, punctuation and separators\n"
    "- text inside stamps, seals, headers, footers, margins, tables and form fields\n"
    "- machine-readable zone (MRZ) lines exactly, including every '<' filler character\n"
    "- visible annotations, handwritten notes and corrections\n"
    "\n"
    "PRESERVE:\n"
    "- the original scripts: keep Arabic text in Arabic, keep Latin text in Latin\n"
    "- the line structure of the page as far as reasonably possible (one line of the document per "
    "line of output)\n"
    "- the original spelling, including apparent errors\n"
    "\n"
    "STRICTLY FORBIDDEN:\n"
    "- do NOT translate anything into another language\n"
    "- do NOT transliterate Arabic into Latin characters or Latin into Arabic characters\n"
    "- do NOT normalize, reformat or standardize names, dates or numbers\n"
    "- do NOT correct spelling, grammar or apparent mistakes\n"
    "- do NOT complete partially visible text\n"
    "- do NOT guess unreadable characters or replace them with plausible alternatives\n"
    "- do NOT infer, summarize, explain or add information that is not visually present\n"
    "- do NOT output commentary, headings, markdown formatting or any description of the document\n"
    "\n"
    "NON-TEXT ELEMENTS:\n"
    "- where a handwritten signature appears, write: [SIGNATURE]\n"
    "- where a stamp appears, write [STAMP] followed by its readable text, if any\n"
    "- where a photograph appears, write: [PHOTO]\n"
    "\n"
    "UNREADABLE CONTENT:\n"
    "- if a character, word or line is genuinely unreadable, write [ILLEGIBLE] in its place\n"
    "- never replace unreadable content with a plausible guess\n"
    "\n"
    "Output the transcription only."
)

print(OCR_PROMPT)
print("\nPrompt characters:", len(OCR_PROMPT))
(DIRS["diagnostics"] / "ocr_prompt.txt").write_text(OCR_PROMPT, encoding="utf-8")

In [ ]:
def ocr_image(image, prompt_text: str = OCR_PROMPT):
    """
    Run ONE Qwen inference call on ONE page image.

    Timing methodology
    ------------------
    * `processor_s` is pure CPU work (tokenization + image preprocessing) -> no sync needed.
    * `h2d_s` wraps the host->device copy and IS followed by a sync, otherwise the copy would be
      attributed to the next measured block.
    * `inference_s` is bracketed by sync-before / sync-after: CUDA is asynchronous, so without the
      trailing sync we would only be timing kernel *launches*, not execution.
    * `decode_s` is CPU-side detokenization.
    We never synchronize between small CPU operations - that would add overhead and measure nothing.
    """
    timings, meta = {}, {}

    t0 = time.perf_counter()
    enc = build_inputs(image, prompt_text)
    timings["processor_s"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    enc = move_to_device(enc)
    cuda_sync()
    timings["h2d_s"] = time.perf_counter() - t0

    input_len = int(enc["input_ids"].shape[-1]) if "input_ids" in enc else None
    if "pixel_values" in enc and hasattr(enc["pixel_values"], "shape"):
        meta["pixel_values_shape"] = list(enc["pixel_values"].shape)

    gen_kwargs = generation_kwargs()
    cuda_sync()
    t0 = time.perf_counter()
    with torch.inference_mode():
        out = MODEL.generate(**enc, **gen_kwargs)
    cuda_sync()
    timings["inference_s"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    gen_ids = out[:, input_len:] if input_len is not None else out
    raw_text = decode_tokens(gen_ids, skip_special_tokens=True)
    raw_text_with_special = decode_tokens(gen_ids, skip_special_tokens=False)
    timings["decode_s"] = time.perf_counter() - t0

    generated_tokens = int(gen_ids.shape[-1])
    meta.update({
        "input_tokens": input_len,
        "generated_tokens": generated_tokens,
        "max_new_tokens": gen_kwargs["max_new_tokens"],
        "hit_token_cap": generated_tokens >= gen_kwargs["max_new_tokens"],
        "tokens_per_second": round(generated_tokens / timings["inference_s"], 2)
        if timings["inference_s"] > 0 else None,
    })

    del enc, out, gen_ids
    return raw_text, raw_text_with_special, timings, meta


print("ocr_image() ready - one call per page, no batching, no multi-page requests.")

## 15. Performance instrumentation

Per-page record schema (written to `raw_ocr_results.jsonl` and `page_timings.csv`):

| group | fields |
|---|---|
| identity | `customer_id`, `document_key`, `document_name`, `pdf_path`, `page_number`, `total_pages`, `file_size_bytes` |
| timing | `pdf_open_s`, `page_discovery_s`, `page_load_s`, `render_s`, `preprocess_s`, `processor_s`, `h2d_s`, `inference_s`, `decode_s`, `write_s`, `total_page_s` |
| tokens | `input_tokens`, `generated_tokens`, `tokens_per_second`, `hit_token_cap` |
| gpu | `gpu_peak_allocated_gib`, `gpu_peak_reserved_gib`, `gpu_allocated_gib`, `gpu_reserved_gib` |
| image | `dpi`, `rendered_width/height`, `final_width/height`, `resized`, `page_rotation` |
| status | `status`, `error_type`, `error_message`, `traceback` |

`pdf_open_s` and `page_discovery_s` are **document-level** measurements (the PDF is opened once and
the page count read once) repeated on each of that document's page records, so a single flat table
stays self-describing. Aggregate rate metrics are computed from `total_page_s`.

In [ ]:
import statistics
from typing import Optional


def percentile(values, q):
    """Linear-interpolation percentile without requiring numpy."""
    vals = sorted(v for v in values if v is not None)
    if not vals:
        return None
    if len(vals) == 1:
        return vals[0]
    pos = (len(vals) - 1) * (q / 100.0)
    low, high = int(pos), min(int(pos) + 1, len(vals) - 1)
    return vals[low] + (vals[high] - vals[low]) * (pos - low)


def compute_aggregates(records, wall_clock_s: Optional[float] = None):
    ok = [r for r in records if r.get("status") == "SUCCESS"]
    failed = [r for r in records if r.get("status") != "SUCCESS"]
    infer = [r["inference_s"] for r in ok if r.get("inference_s") is not None]
    page_t = [r["total_page_s"] for r in ok if r.get("total_page_s") is not None]
    render_t = [r["render_s"] for r in ok if r.get("render_s") is not None]

    agg = {
        "total_documents": len({(r["customer_id"], r["document_name"]) for r in records}),
        "total_customers": len({r["customer_id"] for r in records}),
        "total_pages": len(records),
        "successful_pages": len(ok),
        "failed_pages": len(failed),
        "total_inference_calls": len(infer),
        "success_rate_pct": round(100.0 * len(ok) / len(records), 2) if records else None,
        "avg_inference_s": round(statistics.fmean(infer), 4) if infer else None,
        "median_inference_s": round(statistics.median(infer), 4) if infer else None,
        "p95_inference_s": round(percentile(infer, 95), 4) if infer else None,
        "min_inference_s": round(min(infer), 4) if infer else None,
        "max_inference_s": round(max(infer), 4) if infer else None,
        "stdev_inference_s": round(statistics.pstdev(infer), 4) if len(infer) > 1 else None,
        "avg_page_total_s": round(statistics.fmean(page_t), 4) if page_t else None,
        "median_page_total_s": round(statistics.median(page_t), 4) if page_t else None,
        "p95_page_total_s": round(percentile(page_t, 95), 4) if page_t else None,
        "avg_render_s": round(statistics.fmean(render_t), 4) if render_t else None,
        "total_generated_tokens": sum(r.get("generated_tokens") or 0 for r in ok),
        "avg_generated_tokens": round(statistics.fmean([r.get("generated_tokens") or 0 for r in ok]), 1) if ok else None,
        "avg_input_tokens": round(statistics.fmean([r.get("input_tokens") or 0 for r in ok]), 1) if ok else None,
        "pages_hitting_token_cap": sum(1 for r in ok if r.get("hit_token_cap")),
        "avg_output_chars": round(statistics.fmean([r.get("raw_text_chars") or 0 for r in ok]), 1) if ok else None,
    }

    basis = wall_clock_s if wall_clock_s else (sum(page_t) if page_t else None)
    if basis and len(ok):
        agg["wall_clock_s"] = round(basis, 2)
        agg["pages_per_minute"] = round(len(ok) / (basis / 60.0), 2)
        avg_pages_per_doc = (len(records) / agg["total_documents"]) if agg["total_documents"] else None
        sec_per_page = basis / len(ok)
        agg["estimated_seconds_per_document"] = round(sec_per_page * avg_pages_per_doc, 2) if avg_pages_per_doc else None
        agg["estimated_documents_per_hour"] = (round(3600.0 / agg["estimated_seconds_per_document"], 2)
                                               if agg.get("estimated_seconds_per_document") else None)
        agg["estimated_customers_per_hour"] = (round(agg["estimated_documents_per_hour"] / 5.0, 2)
                                               if agg.get("estimated_documents_per_hour") else None)
        agg["avg_tokens_per_second"] = (round(agg["total_generated_tokens"] / sum(infer), 2)
                                        if infer and sum(infer) > 0 else None)

    # Stage share of total page time - shows where the time actually goes.
    stage_keys = ["render_s", "preprocess_s", "processor_s", "h2d_s", "inference_s", "decode_s", "write_s"]
    total_all = sum(page_t) if page_t else 0.0
    agg["stage_breakdown_pct"] = {
        k: (round(100.0 * sum(r.get(k) or 0.0 for r in ok) / total_all, 2) if total_all else None)
        for k in stage_keys
    }
    return agg


print("Aggregation helpers ready.")

## 16. Error handling — resilient drivers

Three independent recovery levels: **page**, **document**, **customer**. A failure at any level is
recorded (type, message, traceback) and the loop continues. Nothing is swallowed silently: every
exception lands in `errors/errors.jsonl` *and* as a non-`SUCCESS` row in the page table.

Page statuses: `SUCCESS` · `RENDER_ERROR` · `INFERENCE_ERROR` · `WRITE_ERROR` · `SKIPPED_LIMIT`
Document statuses: `OPEN_ERROR` · `EMPTY_PDF` · `PROCESSED`

In [ ]:
ERROR_RECORDS = []


def record_error(customer_id, document_name, page_number, stage, exc, extra=None):
    rec = {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "customer_id": customer_id,
        "document_name": document_name,
        "page_number": page_number,
        "stage": stage,
        "error_type": type(exc).__name__ if isinstance(exc, BaseException) else "Error",
        "error_message": str(exc)[:2000],
        "traceback": traceback.format_exc()[-4000:] if isinstance(exc, BaseException) else "",
    }
    if extra:
        rec.update(extra)
    ERROR_RECORDS.append(rec)
    return rec


def _blank_page_record(customer_id, doc_key, doc_name, pdf_path, page_number, total_pages, file_size):
    """Every page record has the same schema, whatever the outcome."""
    return {
        "run_id": RUN_ID,
        "customer_id": customer_id,
        "document_key": doc_key,
        "document_name": doc_name,
        "pdf_path": str(pdf_path),
        "page_number": page_number,
        "total_pages": total_pages,
        "file_size_bytes": file_size,
        "pdf_open_s": None, "page_discovery_s": None, "page_load_s": None,
        "render_s": None, "preprocess_s": None, "processor_s": None, "h2d_s": None,
        "inference_s": None, "decode_s": None, "write_s": None, "total_page_s": None,
        "input_tokens": None, "generated_tokens": None, "tokens_per_second": None,
        "hit_token_cap": None, "max_new_tokens": CONFIG["max_new_tokens"],
        "gpu_peak_allocated_gib": None, "gpu_peak_reserved_gib": None,
        "gpu_allocated_gib": None, "gpu_reserved_gib": None,
        "dpi": CONFIG["render_dpi"], "rendered_width": None, "rendered_height": None,
        "final_width": None, "final_height": None, "resized": None, "page_rotation": None,
        "preprocessing_applied": bool(CONFIG["enable_optional_preprocessing"]),
        "raw_text_chars": None, "raw_text_lines": None,
        "raw_text": None, "raw_text_file": None,
        "status": "UNKNOWN", "error_type": "", "error_message": "", "traceback": "",
    }


def _page_text_path(customer_id, doc_key, page_number):
    safe_cust = re.sub(r"[^A-Za-z0-9_.-]", "_", str(customer_id))
    d = DIRS["raw_ocr_pages"] / safe_cust
    d.mkdir(parents=True, exist_ok=True)
    return d / f"{doc_key}_page_{page_number:03d}.txt"


def process_pdf(customer_id, doc_key, doc_name, pdf_path, file_size, jsonl_handle,
                page_budget=None):
    """
    Process one PDF: open once, discover pages dynamically, OCR every page in order.
    Returns (page_records, document_status, pages_consumed).
    """
    records = []
    doc = None
    t_open0 = time.perf_counter()
    try:
        doc = PYMUPDF.open(str(pdf_path))
        if getattr(doc, "needs_pass", False) and not doc.authenticate(""):
            raise RuntimeError("PDF is encrypted and requires a password")
        pdf_open_s = time.perf_counter() - t_open0
    except Exception as exc:
        record_error(customer_id, doc_name, None, "pdf_open", exc, {"pdf_path": str(pdf_path)})
        rec = _blank_page_record(customer_id, doc_key, doc_name, pdf_path, None, None, file_size)
        rec.update({"status": "OPEN_ERROR", "error_type": type(exc).__name__,
                    "error_message": str(exc)[:2000], "traceback": traceback.format_exc()[-2000:]})
        _append_jsonl(jsonl_handle, rec)
        records.append(rec)
        return records, "OPEN_ERROR", 0

    t_disc0 = time.perf_counter()
    total_pages = int(doc.page_count)          # dynamic; never assumed
    page_discovery_s = time.perf_counter() - t_disc0

    if total_pages == 0:
        rec = _blank_page_record(customer_id, doc_key, doc_name, pdf_path, None, 0, file_size)
        rec.update({"status": "EMPTY_PDF", "pdf_open_s": pdf_open_s,
                    "page_discovery_s": page_discovery_s,
                    "error_message": "PDF contains zero pages"})
        _append_jsonl(jsonl_handle, rec)
        records.append(rec)
        doc.close()
        return records, "EMPTY_PDF", 0

    n_to_process = total_pages
    if CONFIG["max_pages_per_pdf"]:
        n_to_process = min(n_to_process, int(CONFIG["max_pages_per_pdf"]))
    if page_budget is not None:
        n_to_process = min(n_to_process, max(0, page_budget))

    consumed = 0
    try:
        for page_index in range(total_pages):          # page order preserved, never reordered
            page_number = page_index + 1               # 1-based for reporting only
            if page_index >= n_to_process:
                rec = _blank_page_record(customer_id, doc_key, doc_name, pdf_path,
                                         page_number, total_pages, file_size)
                rec.update({"status": "SKIPPED_LIMIT", "pdf_open_s": pdf_open_s,
                            "page_discovery_s": page_discovery_s,
                            "error_message": "skipped by max_pages_per_pdf / max_total_pages"})
                _append_jsonl(jsonl_handle, rec)
                records.append(rec)
                continue

            rec = _blank_page_record(customer_id, doc_key, doc_name, pdf_path,
                                     page_number, total_pages, file_size)
            rec["pdf_open_s"] = round(pdf_open_s, 5)
            rec["page_discovery_s"] = round(page_discovery_s, 6)
            t_page0 = time.perf_counter()

            if torch is not None and torch.cuda.is_available():
                torch.cuda.reset_peak_memory_stats()

            # ---- render -----------------------------------------------------
            try:
                t0 = time.perf_counter()
                page = doc.load_page(page_index)
                rec["page_load_s"] = round(time.perf_counter() - t0, 5)

                t0 = time.perf_counter()
                image, rmeta = render_page_to_image(page)
                rec["render_s"] = round(time.perf_counter() - t0, 5)
                rec.update({k: rmeta[k] for k in ("dpi", "rendered_width", "rendered_height",
                                                  "final_width", "final_height", "resized",
                                                  "page_rotation")})

                t0 = time.perf_counter()
                image, applied = maybe_preprocess(image)
                rec["preprocess_s"] = round(time.perf_counter() - t0, 5)
                rec["preprocessing_applied"] = applied

                if CONFIG["save_page_images"]:
                    d = DIRS["page_images"] / re.sub(r"[^A-Za-z0-9_.-]", "_", str(customer_id))
                    d.mkdir(parents=True, exist_ok=True)
                    image.save(d / f"{doc_key}_page_{page_number:03d}.png")
            except Exception as exc:
                record_error(customer_id, doc_name, page_number, "render", exc)
                rec.update({"status": "RENDER_ERROR", "error_type": type(exc).__name__,
                            "error_message": str(exc)[:2000],
                            "traceback": traceback.format_exc()[-2000:],
                            "total_page_s": round(time.perf_counter() - t_page0, 5)})
                _append_jsonl(jsonl_handle, rec)
                records.append(rec)
                consumed += 1
                continue

            # ---- inference (exactly one Qwen call per page) ------------------
            try:
                raw_text, raw_special, timings, meta = ocr_image(image)
                rec.update({k: round(v, 5) for k, v in timings.items()})
                rec.update({
                    "input_tokens": meta.get("input_tokens"),
                    "generated_tokens": meta.get("generated_tokens"),
                    "tokens_per_second": meta.get("tokens_per_second"),
                    "hit_token_cap": meta.get("hit_token_cap"),
                    "max_new_tokens": meta.get("max_new_tokens"),
                    "raw_text": raw_text,                       # preserved EXACTLY, unmodified
                    "raw_text_chars": len(raw_text),
                    "raw_text_lines": raw_text.count("\n") + 1 if raw_text else 0,
                    "status": "SUCCESS",
                })
            except Exception as exc:
                record_error(customer_id, doc_name, page_number, "inference", exc)
                rec.update({"status": "INFERENCE_ERROR", "error_type": type(exc).__name__,
                            "error_message": str(exc)[:2000],
                            "traceback": traceback.format_exc()[-2000:],
                            "total_page_s": round(time.perf_counter() - t_page0, 5)})
                _append_jsonl(jsonl_handle, rec)
                records.append(rec)
                consumed += 1
                continue
            finally:
                try:
                    del image
                except Exception:
                    pass

            # ---- persist -----------------------------------------------------
            # Text files first, then GPU stats + total time, THEN the JSONL line - so the
            # streamed JSONL record is complete. write_s covers the per-page text files;
            # the JSONL append itself is the last operation and is added to write_s in the
            # in-memory record only (the line on disk is already serialized by then).
            t_write0 = time.perf_counter()
            try:
                if CONFIG["save_per_page_text_files"]:
                    p = _page_text_path(customer_id, doc_key, page_number)
                    p.write_text(raw_text, encoding="utf-8")
                    rec["raw_text_file"] = str(p)
                    # Raw response WITH special tokens kept separately: nothing is lost.
                    rp = DIRS["raw_ocr_responses"] / p.parent.name
                    rp.mkdir(parents=True, exist_ok=True)
                    (rp / p.name).write_text(raw_special, encoding="utf-8")
                rec["write_s"] = round(time.perf_counter() - t_write0, 5)
            except Exception as exc:
                record_error(customer_id, doc_name, page_number, "write", exc)
                rec.update({"status": "WRITE_ERROR", "error_type": type(exc).__name__,
                            "error_message": str(exc)[:2000],
                            "write_s": round(time.perf_counter() - t_write0, 5)})

            snap = gpu_memory_snapshot(label="page")
            if snap.get("cuda"):
                rec.update({
                    "gpu_peak_allocated_gib": snap["max_allocated_gib"],
                    "gpu_peak_reserved_gib": snap["max_reserved_gib"],
                    "gpu_allocated_gib": snap["allocated_gib"],
                    "gpu_reserved_gib": snap["reserved_gib"],
                })
            rec["total_page_s"] = round(time.perf_counter() - t_page0, 5)

            t_jsonl0 = time.perf_counter()
            try:
                _append_jsonl(jsonl_handle, rec)
            except Exception as exc:
                record_error(customer_id, doc_name, page_number, "write_jsonl", exc)
            rec["write_s"] = round((rec["write_s"] or 0.0) + (time.perf_counter() - t_jsonl0), 5)

            records.append(rec)
            consumed += 1
    finally:
        try:
            doc.close()
        except Exception:
            pass

    return records, "PROCESSED", consumed


def _append_jsonl(handle, record):
    """Append one JSON object per line. ensure_ascii=False keeps Arabic readable on disk."""
    handle.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")
    handle.flush()


print("Resilient drivers ready.")

## 17. Output generation — pipeline execution

This is the run. Results are streamed to `raw_ocr_results.jsonl` **as they are produced**, so a
kernel death mid-run still leaves every completed page on disk.

Customer selection is deterministic: explicit `selected_customers` if provided, otherwise the first
`max_customers` in sorted order (DIAGNOSTIC), otherwise everything (FULL).

In [ ]:
# ---- Deterministic work plan ----------------------------------------------
all_customers = sorted(summary_df["customer_id"].tolist())

if CONFIG["selected_customers"]:
    selected = [c for c in all_customers if c in set(CONFIG["selected_customers"])]
    missing_sel = sorted(set(CONFIG["selected_customers"]) - set(all_customers))
    if missing_sel:
        print("[WARN] selected_customers not found in dataset:", missing_sel)
elif CONFIG["process_all_customers"]:
    selected = all_customers
else:
    selected = all_customers[: int(CONFIG["max_customers"] or 0)]

work_items = existence_df[
    (existence_df["customer_id"].isin(selected)) & (existence_df["status"] == "EXISTS_READABLE")
].sort_values(["customer_id", "document_key"]).reset_index(drop=True)

planned_pages = int(pd.to_numeric(work_items["number_of_pages"], errors="coerce").fillna(0).sum())
if CONFIG["max_pages_per_pdf"]:
    planned_pages = int(sum(min(int(n or 0), int(CONFIG["max_pages_per_pdf"]))
                            for n in work_items["number_of_pages"]))
if CONFIG["max_total_pages"]:
    planned_pages = min(planned_pages, int(CONFIG["max_total_pages"]))

print(f"Run mode          : {CONFIG['run_mode']}")
print(f"Customers selected: {len(selected)} / {len(all_customers)}")
print(f"Documents to OCR  : {len(work_items)}")
print(f"Pages planned     : ~{planned_pages} (= {planned_pages} inference calls)")
print(f"Customers         : {selected[:10]}{' ...' if len(selected) > 10 else ''}")

In [ ]:
PAGE_RECORDS = []
DOCUMENT_RECORDS = []

JSONL_PATH = DIRS["raw_ocr"] / "raw_ocr_results.jsonl"
remaining_budget = int(CONFIG["max_total_pages"]) if CONFIG["max_total_pages"] else None

run_t0 = time.perf_counter()
run_started = datetime.now()
print(f"=== RUN {RUN_ID} started {run_started:%Y-%m-%d %H:%M:%S} ===\n")

with open(JSONL_PATH, "w", encoding="utf-8") as jsonl_handle:
    for i, item in work_items.iterrows():
        cust, key, name = item["customer_id"], item["document_key"], item["document_name"]
        doc_t0 = time.perf_counter()
        if remaining_budget is not None and remaining_budget <= 0:
            print("[BUDGET] max_total_pages reached - stopping.")
            break
        try:
            recs, doc_status, consumed = process_pdf(
                cust, key, name, item["path"], int(item["file_size_bytes"]),
                jsonl_handle, page_budget=remaining_budget,
            )
        except Exception as exc:                      # customer/document-level safety net
            record_error(cust, name, None, "document_driver", exc)
            recs, doc_status, consumed = [], "DRIVER_ERROR", 0

        PAGE_RECORDS.extend(recs)
        if remaining_budget is not None:
            remaining_budget -= consumed

        ok = sum(1 for r in recs if r["status"] == "SUCCESS")
        doc_elapsed = time.perf_counter() - doc_t0
        DOCUMENT_RECORDS.append({
            "run_id": RUN_ID, "customer_id": cust, "document_key": key, "document_name": name,
            "pdf_path": item["path"], "total_pages": item["number_of_pages"],
            "pages_processed": consumed, "pages_success": ok, "pages_failed": consumed - ok,
            "document_status": doc_status, "document_elapsed_s": round(doc_elapsed, 3),
        })

        if (i + 1) % max(1, int(CONFIG["progress_every"])) == 0:
            done_pages = len([r for r in PAGE_RECORDS if r["status"] == "SUCCESS"])
            elapsed = time.perf_counter() - run_t0
            rate = done_pages / (elapsed / 60.0) if elapsed > 0 else 0
            print(f"[{i+1}/{len(work_items)}] {cust} / {name}: "
                  f"{ok}/{consumed} pages OK in {doc_elapsed:.1f}s "
                  f"| cumulative {done_pages} pages, {rate:.1f} pages/min")

RUN_WALL_CLOCK_S = time.perf_counter() - run_t0
print(f"\n=== RUN FINISHED in {RUN_WALL_CLOCK_S/60:.2f} min "
      f"({len(PAGE_RECORDS)} page records, {len(ERROR_RECORDS)} errors) ===")

In [ ]:
# ---- Materialize all output artefacts --------------------------------------
pages_df = pd.DataFrame(PAGE_RECORDS)
docs_df = pd.DataFrame(DOCUMENT_RECORDS)
errors_df = pd.DataFrame(ERROR_RECORDS)

# 1) CSV of raw OCR (text inlined; newlines preserved by the CSV quoting rules)
csv_path = DIRS["raw_ocr"] / "raw_ocr_results.csv"
if len(pages_df):
    pages_df.to_csv(csv_path, index=False, encoding="utf-8-sig")

# 2) Human-readable concatenated TXT
txt_path = DIRS["raw_ocr"] / "raw_ocr_results.txt"
with open(txt_path, "w", encoding="utf-8") as fh:
    for r in PAGE_RECORDS:
        fh.write("=" * 78 + "\n")
        fh.write(f"customer_id : {r['customer_id']}\n")
        fh.write(f"document    : {r['document_name']}\n")
        fh.write(f"page        : {r['page_number']} / {r['total_pages']}\n")
        fh.write(f"status      : {r['status']}\n")
        if r.get("error_message"):
            fh.write(f"error       : {r['error_type']}: {r['error_message']}\n")
        fh.write("-" * 78 + "\n")
        fh.write((r.get("raw_text") or "") + "\n\n")

# 3) Timings (no text columns -> small and fast to load)
timing_cols = [c for c in pages_df.columns if c not in ("raw_text", "traceback")] if len(pages_df) else []
if len(pages_df):
    pages_df[timing_cols].to_csv(DIRS["performance"] / "page_timings.csv", index=False, encoding="utf-8-sig")
    docs_df.to_csv(DIRS["performance"] / "document_summary.csv", index=False, encoding="utf-8-sig")

# 4) Errors
if len(errors_df):
    errors_df.to_csv(DIRS["errors"] / "errors.csv", index=False, encoding="utf-8-sig")
with open(DIRS["errors"] / "errors.jsonl", "w", encoding="utf-8") as fh:
    for e in ERROR_RECORDS:
        fh.write(json.dumps(e, ensure_ascii=False, default=str) + "\n")

print("Artefacts written:")
for p in [JSONL_PATH, csv_path, txt_path,
          DIRS["performance"] / "page_timings.csv",
          DIRS["performance"] / "document_summary.csv",
          DIRS["errors"] / "errors.jsonl"]:
    print(f"  {'OK ' if Path(p).exists() else '-- '} {p}")
print(f"  OK  {DIRS['raw_ocr_pages']}/<customer>/<doc>_page_NNN.txt  (per-page text)")
print(f"  OK  {DIRS['raw_ocr_responses']}/  (raw responses incl. special tokens)")

## 18. Performance summary

Aggregates over **measured** pages only. Warmup and model loading are reported separately and are
excluded from every statistic below.

In [ ]:
AGGREGATES = compute_aggregates(PAGE_RECORDS, wall_clock_s=RUN_WALL_CLOCK_S)

PERFORMANCE_REPORT = {
    "run_id": RUN_ID,
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "config": {k: (str(v) if isinstance(v, Path) else v) for k, v in CONFIG.items()},
    "environment": ENV_REPORT,
    "gpu": GPU_REPORT,
    "model": {k: str(v) for k, v in MODEL_LOAD_REPORT.items() if k != "attempts"},
    "processor": {"class": type(PROCESSOR).__name__, "strategy": PROCESSOR_STRATEGY,
                  "init_time_s": round(PROCESSOR_INIT_TIME_S, 3),
                  "chat_template": HAS_CHAT_TEMPLATE},
    "phases": {
        "model_loading_s": round(MODEL_LOAD_TIME_S, 3),
        "processor_init_s": round(PROCESSOR_INIT_TIME_S, 3),
        "warmup_s": round(WARMUP_TOTAL_S, 3),
        "measured_inference_wall_clock_s": round(RUN_WALL_CLOCK_S, 3),
        "pdf_validation_s": round(validation_elapsed, 3),
    },
    "warmup_detail": WARMUP_RESULTS,
    "aggregates": AGGREGATES,
    "gpu_snapshots": {
        "before_model_load": GPU_BEFORE_LOAD,
        "after_model_load": GPU_AFTER_LOAD,
        "after_warmup": GPU_AFTER_WARMUP,
        "after_run": gpu_memory_snapshot(label="after_run"),
    },
}

(DIRS["performance"] / "performance_report.json").write_text(
    json.dumps(PERFORMANCE_REPORT, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

print("=" * 78)
print("PHASE TIMINGS (warmup strictly excluded from measured statistics)")
print("=" * 78)
for k, v in PERFORMANCE_REPORT["phases"].items():
    print(f"  {k:<36}: {v:>10.2f} s")

print("\n" + "=" * 78)
print("AGGREGATE PERFORMANCE")
print("=" * 78)
for k, v in AGGREGATES.items():
    if k == "stage_breakdown_pct":
        continue
    print(f"  {k:<36}: {v}")

print("\n  stage share of total page time (%):")
for k, v in (AGGREGATES.get("stage_breakdown_pct") or {}).items():
    print(f"    {k:<20}: {v}")

## 19. Diagnostic visualizations

Six engineering plots — distributions and counts only. Not a dashboard.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

ok_df = pages_df[pages_df["status"] == "SUCCESS"] if len(pages_df) else pages_df

if not len(ok_df):
    print("[INFO] No successful pages - nothing to plot.")
else:
    fig, axes = plt.subplots(2, 3, figsize=(17, 9))
    fig.suptitle(f"KYC OCR Stage 1 diagnostics - run {RUN_ID}", fontsize=13)

    axes[0, 0].hist(ok_df["inference_s"].dropna(), bins=25, color="#2b6cb0", edgecolor="white")
    axes[0, 0].set_title("Inference time per page (s)")
    axes[0, 0].set_xlabel("seconds"); axes[0, 0].set_ylabel("pages")
    _m = ok_df["inference_s"].median()
    axes[0, 0].axvline(_m, color="crimson", ls="--", label=f"median {_m:.2f}s")
    axes[0, 0].legend()

    axes[0, 1].hist(ok_df["total_page_s"].dropna(), bins=25, color="#2f855a", edgecolor="white")
    axes[0, 1].set_title("Total page processing time (s)")
    axes[0, 1].set_xlabel("seconds"); axes[0, 1].set_ylabel("pages")

    by_doc = pages_df.groupby("document_name")["status"].value_counts().unstack(fill_value=0)
    by_doc.plot(kind="barh", stacked=True, ax=axes[0, 2], colormap="Set2", legend=True)
    axes[0, 2].set_title("Pages processed by document type")
    axes[0, 2].set_xlabel("pages"); axes[0, 2].set_ylabel("")
    axes[0, 2].tick_params(axis="y", labelsize=8)

    status_counts = pages_df["status"].value_counts()
    axes[1, 0].bar(status_counts.index.astype(str), status_counts.values, color="#b7791f")
    axes[1, 0].set_title("Page outcomes / error counts")
    axes[1, 0].tick_params(axis="x", rotation=30, labelsize=8)
    for i, v in enumerate(status_counts.values):
        axes[1, 0].text(i, v, str(v), ha="center", va="bottom", fontsize=8)

    if ok_df["gpu_peak_allocated_gib"].notna().any():
        axes[1, 1].plot(range(1, len(ok_df) + 1), ok_df["gpu_peak_allocated_gib"].values,
                        label="peak allocated", color="#805ad5")
        axes[1, 1].plot(range(1, len(ok_df) + 1), ok_df["gpu_reserved_gib"].values,
                        label="reserved", color="#dd6b20", alpha=0.7)
        axes[1, 1].set_title("GPU memory per page (GiB)")
        axes[1, 1].set_xlabel("page index"); axes[1, 1].legend(fontsize=8)
    else:
        axes[1, 1].text(0.5, 0.5, "no GPU memory data", ha="center"); axes[1, 1].set_axis_off()

    axes[1, 2].hist(ok_df["raw_text_chars"].dropna(), bins=25, color="#319795", edgecolor="white")
    axes[1, 2].set_title("OCR output length (characters)")
    axes[1, 2].set_xlabel("characters"); axes[1, 2].set_ylabel("pages")

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(DIRS["diagnostics"] / "diagnostics.png", dpi=130, bbox_inches="tight")
    plt.show()
    print("Saved:", DIRS["diagnostics"] / "diagnostics.png")

    # Correlation sanity check: does output length drive latency? (It should.)
    if ok_df["generated_tokens"].notna().any():
        corr = ok_df[["generated_tokens", "inference_s"]].corr().iloc[0, 1]
        print(f"\ncorr(generated_tokens, inference_s) = {corr:.3f}")
        print("A high correlation confirms decode length - not image size - dominates latency,")
        print("which is the main lever for tuning max_new_tokens.")

## 20. Final run summary

Everything needed to decide whether this baseline is trustworthy, and what stage 2 (structured
extraction) should be designed against.

In [ ]:
ok_n = int((pages_df["status"] == "SUCCESS").sum()) if len(pages_df) else 0
fail_n = len(pages_df) - ok_n if len(pages_df) else 0

print("=" * 78)
print(f"KYC OCR STAGE 1 - FINAL SUMMARY   (run {RUN_ID})")
print("=" * 78)
print(f"Dataset          : {ZIP_PATH.name}  ->  {DATASET_ROOT}")
print(f"Customers found  : {len(customer_dirs)}")
print(f"Complete folders : {int(summary_df['complete_folder'].sum())} / {len(summary_df)} (5/5 documents)")
print(f"Required PDFs    : present={int(existence_df['exists'].sum())} "
      f"readable={int((existence_df['status'] == 'EXISTS_READABLE').sum())} "
      f"unreadable={int((existence_df['status'] == 'EXISTS_UNREADABLE').sum())} "
      f"missing={int((existence_df['status'] == 'MISSING').sum())}")
print()
print(f"Run mode         : {CONFIG['run_mode']}  (customers processed: {len(selected)})")
print(f"Documents OCR'd  : {len(docs_df)}")
print(f"Pages            : {len(pages_df)}  (success {ok_n}, failed/skipped {fail_n})")
print(f"Inference calls  : {AGGREGATES.get('total_inference_calls')}  (exactly 1 per page)")
print()
print(f"Model            : {MODEL_LOAD_REPORT['model_class']} @ {MODEL_LOAD_REPORT['dtype']} "
      f"on {MODEL_LOAD_REPORT['device']} (attn={MODEL_LOAD_REPORT['attn_implementation']})")
print(f"Processor        : {type(PROCESSOR).__name__} (strategy: {PROCESSOR_STRATEGY})")
print(f"Render           : {CONFIG['render_dpi']} DPI, long side <= {CONFIG['max_image_long_side']} px, "
      f"preprocessing={'ON' if CONFIG['enable_optional_preprocessing'] else 'OFF (raw baseline)'}")
print(f"Generation       : max_new_tokens={CONFIG['max_new_tokens']}, do_sample={CONFIG['do_sample']}")
print()
print(f"Model load       : {MODEL_LOAD_TIME_S:.1f}s   Warmup: {WARMUP_TOTAL_S:.1f}s (excluded)")
print(f"Measured run     : {RUN_WALL_CLOCK_S/60:.2f} min")
print(f"Inference/page   : avg={AGGREGATES.get('avg_inference_s')}s "
      f"median={AGGREGATES.get('median_inference_s')}s p95={AGGREGATES.get('p95_inference_s')}s "
      f"min={AGGREGATES.get('min_inference_s')}s max={AGGREGATES.get('max_inference_s')}s")
print(f"Throughput       : {AGGREGATES.get('pages_per_minute')} pages/min "
      f"-> ~{AGGREGATES.get('estimated_documents_per_hour')} documents/hour "
      f"(~{AGGREGATES.get('estimated_customers_per_hour')} customers/hour)")
print(f"Tokens           : avg_in={AGGREGATES.get('avg_input_tokens')} "
      f"avg_out={AGGREGATES.get('avg_generated_tokens')} "
      f"({AGGREGATES.get('avg_tokens_per_second')} tok/s) "
      f"| pages hitting the cap: {AGGREGATES.get('pages_hitting_token_cap')}")
print(f"Errors           : {len(ERROR_RECORDS)}")
if len(errors_df):
    print(errors_df.groupby(["stage", "error_type"]).size().to_string())

if AGGREGATES.get("pages_hitting_token_cap"):
    print(f"\n[ACTION] {AGGREGATES['pages_hitting_token_cap']} page(s) hit max_new_tokens="
          f"{CONFIG['max_new_tokens']} - their transcription is TRUNCATED. Inspect those pages "
          "before raising the cap (a looping model also hits the cap).")

FINAL_SUMMARY = {
    "run_id": RUN_ID,
    "dataset": {"zip": str(ZIP_PATH), "root": str(DATASET_ROOT), "customers": len(customer_dirs)},
    "selection": {"mode": CONFIG["run_mode"], "customers_processed": selected},
    "counts": {"documents": len(docs_df), "pages": len(pages_df), "success": ok_n,
               "failed_or_skipped": fail_n, "errors": len(ERROR_RECORDS)},
    "aggregates": AGGREGATES,
    "phases": PERFORMANCE_REPORT["phases"],
    "outputs": {k: str(v) for k, v in DIRS.items()},
}
(DIRS["root"] / "final_run_summary.json").write_text(
    json.dumps(FINAL_SUMMARY, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

print("\nOutput tree:")
for p in sorted(OUT.rglob("*")):
    if p.is_file() and len(str(p.relative_to(OUT)).split(os.sep)) <= 2:
        print(f"  {p.relative_to(OUT)}  ({p.stat().st_size/1024:.1f} KiB)")

print("\nNEXT STAGE INPUT: outputs/raw_ocr/raw_ocr_results.jsonl")
print("Reconstruct customer -> document -> page -> OCR with:")
print("  df = pd.read_json('outputs/raw_ocr/raw_ocr_results.jsonl', lines=True)")
print("  df.sort_values(['customer_id','document_name','page_number'])")

### Sample output inspection

A quick look at what the model actually produced — the only real acceptance test for this stage.
Read a handful of these side by side with the source scans before designing structured extraction.

In [ ]:
sample = ok_df.head(3) if len(ok_df) else pages_df.head(0)
for _, r in sample.iterrows():
    print("=" * 78)
    print(f"{r['customer_id']} | {r['document_name']} | page {r['page_number']}/{r['total_pages']}")
    print(f"inference={r['inference_s']}s  tokens={r['generated_tokens']}  chars={r['raw_text_chars']}")
    print("-" * 78)
    text = r["raw_text"] or ""
    print(text[:1500] + ("\n... [truncated for display]" if len(text) > 1500 else ""))
    print()

if len(ok_df):
    short = ok_df[ok_df["raw_text_chars"] < 40]
    if len(short):
        print(f"[REVIEW] {len(short)} page(s) produced fewer than 40 characters - likely blank pages, "
              "failed reads or model refusals. Inspect them manually:")
        print(short[["customer_id", "document_name", "page_number", "raw_text_chars"]].to_string(index=False))